# Prior-revision research: baf_base

Training/validation only. No calibration or final predictions. Adaptive development, not independent confirmation. Exact curvature, simple controls and known related work are evaluated explicitly.

In [ ]:
from pathlib import Path
import os,sys,json
os.environ["OMP_NUM_THREADS"]="2"
os.environ["OPENBLAS_NUM_THREADS"]="2"
root=Path("/kaggle/working/v7_experiment");root.mkdir(exist_ok=True)
files={'altcboost.py': '"""ALTCBoost — experimental Alternating Linear, Tree and Cross boosting classifier.\n\nResearch prototype for numeric tabular binary classification. Not claimed to be\nnovel relative to all published literature, or universally better than XGBoost.\n\nAlgorithm:\n  1. Fit a regularized logistic (global linear) prior.\n  2. Correct logits using 2nd-order (Newton/IRLS) regression trees.\n  3. After every tree, greedily choose one explicit pairwise interaction using\n     second-order gain, update the logit margin with an L2-regularized step.\n  4. Use a held-out validation set for early stopping.\n\nNo XGBoost used in the proposed model itself.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Optional\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.tree import DecisionTreeRegressor\nfrom sklearn.model_selection import train_test_split\nfrom scipy.special import expit, logit\nfrom sklearn.metrics import log_loss\n\n\nclass ALTCBoostClassifier:\n    def __init__(self, n_estimators=90, learning_rate=0.08, max_leaf_nodes=9,\n                 min_samples_leaf=35, linear_C=1.0, pair_penalty=10.0,\n                 pair_rate=0.35, early_stopping_rounds=16, random_state=42,\n                 use_cross=True):\n        self.n_estimators=n_estimators\n        self.learning_rate=learning_rate\n        self.max_leaf_nodes=max_leaf_nodes\n        self.min_samples_leaf=min_samples_leaf\n        self.linear_C=linear_C\n        self.pair_penalty=pair_penalty\n        self.pair_rate=pair_rate\n        self.early_stopping_rounds=early_stopping_rounds\n        self.random_state=random_state\n        self.use_cross=use_cross\n\n    def _features(self, X):\n        X=np.asarray(X,dtype=float)\n        if X.ndim != 2: raise ValueError(\'X must be a 2D numeric array\')\n        return np.clip(self.scaler_.transform(X),-6,6)\n\n    def _cross_matrix(self,X):\n        i,j=self.pairs_[:,0],self.pairs_[:,1]\n        C=X[:,i]*X[:,j]\n        # subtract train-time means and divide train-time standard deviations\n        return np.clip((C-self.cross_means_)/self.cross_stds_,-10,10)\n\n    @staticmethod\n    def _loss(y, logits):\n        return float(np.mean(np.logaddexp(0, logits)-y*logits))\n\n    def fit(self, X, y, X_val=None, y_val=None):\n        X=np.asarray(X,dtype=float)\n        y=np.asarray(y,dtype=int)\n        if not np.all(np.isfinite(X)): raise ValueError(\'Only finite numerical X supported in prototype.\')\n        if X_val is None:\n            X,X_val,y,y_val=train_test_split(X,y,test_size=0.22,stratify=y,random_state=self.random_state)\n        else:\n            X_val=np.asarray(X_val,dtype=float)\n            y_val=np.asarray(y_val,dtype=int)\n        if np.unique(y).size!=2: raise ValueError(\'binary y with both classes required\')\n        self.scaler_=StandardScaler().fit(X)\n        Xt=self._features(X); Xv=self._features(X_val)\n        self.linear_=LogisticRegression(C=self.linear_C, solver=\'lbfgs\',max_iter=500,random_state=self.random_state).fit(Xt,y)\n        F=self.linear_.decision_function(Xt).astype(float)\n        Fv=self.linear_.decision_function(Xv).astype(float)\n        self.steps_=[]\n        self.best_iteration_=0\n        best_loss=self._loss(y_val,Fv)\n        best_idx=0\n        patience=0\n        self.pairs_=np.array([(i,j) for i in range(X.shape[1]) for j in range(i+1,X.shape[1])],dtype=int).reshape(-1,2)\n        if self.pairs_.size and self.use_cross:\n            i,j=self.pairs_[:,0],self.pairs_[:,1]\n            Ct0=Xt[:,i]*Xt[:,j]\n            self.cross_means_=Ct0.mean(axis=0)\n            self.cross_stds_=np.maximum(Ct0.std(axis=0),.05)\n            Ct=self._cross_matrix(Xt); Cv=self._cross_matrix(Xv)\n        else:\n            self.cross_means_=np.array([])\n            self.cross_stds_=np.array([])\n            Ct=Cv=None\n        for t in range(self.n_estimators):\n            p=expit(np.clip(F,-25,25))\n            g=y-p\n            h=np.maximum(p*(1-p),0.002)\n            newton=np.clip(g/h,-15,15)\n            tree=DecisionTreeRegressor(max_leaf_nodes=self.max_leaf_nodes,\n                 min_samples_leaf=self.min_samples_leaf,random_state=self.random_state+t)\n            tree.fit(Xt,newton,sample_weight=h)\n            dtr=np.clip(tree.predict(Xt),-5,5)*self.learning_rate\n            dval=np.clip(tree.predict(Xv),-5,5)*self.learning_rate\n            F+=dtr; Fv+=dval\n            cross_info=None\n            if Ct is not None and Ct.shape[1]:\n                p=expit(np.clip(F,-25,25))\n                g=y-p; h=np.maximum(p*(1-p),0.002)\n                # one-coordinate Newton gain, optimized on the current residual\n                numerator=Ct.T@g\n                denominator=(Ct**2).T@h+self.pair_penalty\n                scores=numerator*numerator/denominator\n                j=int(np.argmax(scores))\n                coeff=np.clip(numerator[j]/denominator[j],-3,3)*self.learning_rate*self.pair_rate\n                F+=coeff*Ct[:,j]\n                Fv+=coeff*Cv[:,j]\n                cross_info=(j, float(coeff))\n            self.steps_.append((tree,cross_info))\n            val_loss=self._loss(y_val,Fv)\n            if val_loss<best_loss-1e-5:\n                best_loss=val_loss\n                best_idx=t+1\n                patience=0\n            else:\n                patience+=1\n            if patience >= self.early_stopping_rounds:\n                break\n        self.best_iteration_=best_idx\n        self.validation_logloss_=best_loss\n        return self\n\n    def decision_function(self, X):\n        Xt=self._features(X)\n        F=self.linear_.decision_function(Xt).astype(float)\n        Ct=self._cross_matrix(Xt) if self.use_cross and self.pairs_.size else None\n        for tree, cross_info in self.steps_[:self.best_iteration_]:\n            F+=self.learning_rate*np.clip(tree.predict(Xt),-5,5)\n            if cross_info is not None:\n                j,coeff=cross_info\n                F+=coeff*Ct[:,j]\n        return F\n\n    def predict_proba(self,X):\n        p=expit(np.clip(self.decision_function(X),-35,35))\n        return np.column_stack((1-p,p))\n\n    def predict(self,X):\n        return (self.predict_proba(X)[:,1]>.5).astype(int)\n', 'altcboost_v2.py': '"""ALTCBoost 2.0 research prototype: exact compiled inference + FPR operating point.\n\nFit uses the original ALTCBoost algorithm (same training and ranking). Compiles\ntrained coefficients and tree structure into Numba CPU kernels for single-event\ninference; builds optional safe branch-and-bound classification using tree leaf\nbounds. Calibrates an empirical operating threshold from *separate* labeled\ncalibration data. No claim of population FPR guarantee or universal superiority.\n"""\nfrom __future__ import annotations\nimport numpy as np\nfrom scipy.special import expit, logit\nfrom scipy.stats import beta\nfrom numba import njit\nfrom altcboost import ALTCBoostClassifier\n\n\n@njit(cache=True)\ndef _score(X, means, scales, coef, intercept, ci, cj, cm, cs, cc,\n           offsets, left, right, feat, threshold, vals, lr):\n    out = np.empty(X.shape[0], dtype=np.float64)\n    for row in range(X.shape[0]):\n        z = np.empty(X.shape[1], dtype=np.float64)\n        m = intercept\n        for j in range(X.shape[1]):\n            v = (X[row,j] - means[j]) / scales[j]\n            z[j] = min(6.0, max(-6.0, v))\n            m += coef[j] * z[j]\n        for j in range(len(ci)):\n            c = (z[ci[j]] * z[cj[j]] - cm[j]) / cs[j]\n            m += cc[j] * min(10.0, max(-10.0, c))\n        for t in range(len(offsets)-1):\n            base = offsets[t]\n            node = base\n            while left[node] != -1:\n                # sklearn converts tree inputs to float32 before comparing splits.\n                if np.float32(z[feat[node]]) <= threshold[node]:\n                    node = base + left[node]\n                else:\n                    node = base + right[node]\n            m += lr * min(5.0,max(-5.0,vals[node]))\n        out[row] = m\n    return out\n\n\n@njit(cache=True)\ndef _classify_early(X, decision_logit, means, scales, coef, intercept,\n                    ci, cj, cm, cs, cc, offsets, left, right, feat,\n                    threshold, vals, lr, suffix_min, suffix_max):\n    out = np.zeros(X.shape[0], dtype=np.uint8)\n    trees_evaluated = np.zeros(X.shape[0],dtype=np.int32)\n    for row in range(X.shape[0]):\n        z = np.empty(X.shape[1], dtype=np.float64)\n        m = intercept\n        for j in range(X.shape[1]):\n            v = (X[row,j] - means[j]) / scales[j]\n            z[j] = min(6.0,max(-6.0,v))\n            m += coef[j]*z[j]\n        for j in range(len(ci)):\n            c = (z[ci[j]]*z[cj[j]]-cm[j])/cs[j]\n            m += cc[j]*min(10.0,max(-10.0,c))\n        for t in range(len(offsets)-1):\n            if m+suffix_max[t] < decision_logit:\n                break\n            if m+suffix_min[t] >= decision_logit:\n                out[row]=1\n                break\n            base=offsets[t]\n            node=base\n            while left[node]!=-1:\n                if np.float32(z[feat[node]]) <= threshold[node]:\n                    node=base+left[node]\n                else:\n                    node=base+right[node]\n            m += lr*min(5.0,max(-5.0,vals[node]))\n            trees_evaluated[row]+=1\n        else:\n            out[row]=int(m>=decision_logit)\n    return out,trees_evaluated\n\n\nclass ALTCBoostV2(ALTCBoostClassifier):\n    """Numerically compatible accelerated prediction of an ALTCBoost model."""\n    def fit(self, X, y, X_val=None, y_val=None):\n        super().fit(X,y,X_val,y_val)\n        for name in list(vars(self)):\n            if name == \'decision_threshold_\' or name.startswith(\'calibration_\'):\n                delattr(self, name)\n        return self.compile()\n\n    def compile(self):\n        self._means=np.asarray(self.scaler_.mean_,dtype=np.float64)\n        self._scales=np.asarray(self.scaler_.scale_,dtype=np.float64)\n        self._coef=np.asarray(self.linear_.coef_[0],dtype=np.float64)\n        self._intercept=float(self.linear_.intercept_[0])\n        # Collapse repeated pairwise updates into a single coefficient / unique pair.\n        agg={}\n        for tree,cross in self.steps_[:self.best_iteration_]:\n            if cross is not None:\n                idx, delta=cross\n                agg[idx]=agg.get(idx,0.)+delta\n        ids=np.array(list(agg),dtype=np.int32)\n        self._ci=np.asarray(self.pairs_[ids,0] if len(ids) else [],dtype=np.int32)\n        self._cj=np.asarray(self.pairs_[ids,1] if len(ids) else [],dtype=np.int32)\n        self._cm=np.asarray(self.cross_means_[ids] if len(ids) else [],dtype=np.float64)\n        self._cs=np.asarray(self.cross_stds_[ids] if len(ids) else [],dtype=np.float64)\n        self._cc=np.asarray(list(agg.values()),dtype=np.float64)\n        offsets=[0]; left=[];right=[]; feat=[];thr=[];vals=[]\n        mins=[];maxs=[]\n        for tree,_ in self.steps_[:self.best_iteration_]:\n            nodes=tree.tree_\n            l=nodes.children_left.astype(np.int32)\n            r=nodes.children_right.astype(np.int32)\n            v=nodes.value[:,0,0].astype(np.float64)\n            left.extend(l);right.extend(r)\n            feat.extend(nodes.feature.astype(np.int32))\n            thr.extend(nodes.threshold.astype(np.float64));vals.extend(v)\n            offsets.append(offsets[-1]+len(v))\n            leafvals=np.clip(v[l==-1],-5.,5.)*self.learning_rate\n            mins.append(leafvals.min());maxs.append(leafvals.max())\n        self._offsets=np.asarray(offsets,dtype=np.int32)\n        self._left=np.asarray(left,dtype=np.int32)\n        self._right=np.asarray(right,dtype=np.int32)\n        self._feat=np.asarray(feat,dtype=np.int32)\n        self._thr=np.asarray(thr,dtype=np.float64)\n        self._vals=np.asarray(vals,dtype=np.float64)\n        self._suffix_min=np.r_[np.cumsum(mins[::-1])[::-1],0.].astype(np.float64)\n        self._suffix_max=np.r_[np.cumsum(maxs[::-1])[::-1],0.].astype(np.float64)\n        return self\n\n    def _args(self):\n        return (self._means,self._scales,self._coef,self._intercept,\n                self._ci,self._cj,self._cm,self._cs,self._cc,self._offsets,\n                self._left,self._right,self._feat,self._thr,self._vals,float(self.learning_rate))\n\n    def decision_function(self,X):\n        X=np.asarray(X,dtype=np.float64)\n        if X.ndim != 2 or X.shape[1] != len(self._means) or not np.isfinite(X).all():\n            raise ValueError(\'Input must be finite with the fitted feature count\')\n        return _score(X,*self._args())\n\n    def predict_proba(self,X):\n        p=expit(np.clip(self.decision_function(X),-35.,35.))\n        return np.column_stack((1.-p,p))\n\n    def fit_threshold(self, X_cal, y_cal, max_fpr=0.01, confidence=None):\n        """Conservative empirical FPR cap, not a future false-positive guarantee.\n\n        Use a calibration set distinct from train, early-stop selection, and test.\n        """\n        if not 0.<=max_fpr<1.:\n            raise ValueError(\'max_fpr must be between 0 and 1\')\n        labels=np.asarray(y_cal,dtype=np.int32)\n        neg=np.sort(self.predict_proba(X_cal)[labels==0,1])[::-1]\n        if len(neg)==0: raise ValueError(\'No negative calibration examples\')\n        allow=int(np.floor(max_fpr*len(neg)))\n        if confidence is not None:\n            if not 0. < confidence < 1.:\n                raise ValueError(\'confidence must be in (0,1)\')\n            # One-sided order-statistic tolerance bound using Beta tails.\n            # Requires calibration negatives representative of future cases.\n            permitted=[k for k in range(allow+1)\n                       if float(beta.ppf(confidence,k+1,len(neg)-k)) <= max_fpr]\n            if not permitted:\n                raise ValueError(\'Not enough negative calibration samples for a\'\n                                 \' statistically supported FPR bound at this confidence\')\n            allow=max(permitted)\n        if allow>=len(neg):\n            raise ValueError(\'max_fpr too large\')\n        self.decision_threshold_=float(np.nextafter(neg[allow],np.inf))\n        self.calibration_n_negative_=len(neg)\n        self.calibration_max_fpr_=max_fpr\n        self.calibration_confidence_=confidence\n        self.calibration_allowed_false_positives_=allow\n        return self\n\n    def predict_flags(self,X,threshold=None):\n        if threshold is None:\n            threshold=getattr(self,\'decision_threshold_\',None)\n        if threshold is None:\n            raise ValueError(\'fit_threshold or pass a threshold before classification\')\n        if not 0.<threshold<1.: raise ValueError(\'threshold should be within (0,1)\')\n        X=np.asarray(X,dtype=np.float64)\n        if X.ndim != 2 or X.shape[1] != len(self._means) or not np.isfinite(X).all():\n            raise ValueError(\'Input must be finite with the fitted feature count\')\n        return _classify_early(X,float(logit(threshold)),*self._args(),\n                               self._suffix_min,self._suffix_max)\n', 'altcboost_v3.py': '"""ALTCBoost V3: a *research prototype* of hard-negative-aware linear/tree/cross boosting.\n\nResearch ingredients motivated by partial ROC-AUC and Neyman-Pearson literature:\n    - Explicit emphasis on the current high-scoring negative examples in boosted\n      Newton updates (adaptive weights; *not* exact partial AUC minimization).\n    - Validation model selection trades average precision and standardized\n      partial ROC-AUC at a small false-positive rate.\n    - Independent calibration via the inherited V2 method.\n    - Compact compiled Numba inference and exact decision early exit from V2.\n\nThis is an experimental combination of known ideas, NOT a claim of a novel\npublished algorithm or superiority to strong established baselines.\n"""\nfrom __future__ import annotations\nimport numpy as np\nfrom scipy.special import expit\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.tree import DecisionTreeRegressor\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom altcboost_v2 import ALTCBoostV2\n\n\nclass ALTCBoostV3(ALTCBoostV2):\n    def __init__(self, *, hard_negative_strength=2.5,\n                 hard_negative_fraction=0.08,\n                 hard_negative_temperature=0.35,\n                 hard_negative_warmup=12,\n                 low_fpr_validation=0.05,\n                 validation_ap_weight=0.5,\n                 n_estimators=90, learning_rate=0.08, max_leaf_nodes=9,\n                 min_samples_leaf=35, linear_C=1.0, pair_penalty=10.0,\n                 pair_rate=0.35, early_stopping_rounds=16, random_state=42,\n                 use_cross=True):\n        super().__init__(n_estimators=n_estimators,\n            learning_rate=learning_rate,max_leaf_nodes=max_leaf_nodes,\n            min_samples_leaf=min_samples_leaf,linear_C=linear_C,\n            pair_penalty=pair_penalty,pair_rate=pair_rate,\n            early_stopping_rounds=early_stopping_rounds,\n            random_state=random_state,use_cross=use_cross)\n        if not (0 < hard_negative_fraction <= 0.5):\n            raise ValueError(\'hard_negative_fraction must be in (0, .5]\')\n        if hard_negative_strength < 0 or hard_negative_temperature <= 0:\n            raise ValueError(\'invalid hard negative weighting\')\n        if not 0 < low_fpr_validation <= 1:\n            raise ValueError(\'low_fpr_validation must be in (0,1]\')\n        if not 0 <= validation_ap_weight <= 1:\n            raise ValueError(\'validation_ap_weight must be in [0,1]\')\n        self.hard_negative_strength=float(hard_negative_strength)\n        self.hard_negative_fraction=float(hard_negative_fraction)\n        self.hard_negative_temperature=float(hard_negative_temperature)\n        self.hard_negative_warmup=int(hard_negative_warmup)\n        self.low_fpr_validation=float(low_fpr_validation)\n        self.validation_ap_weight=float(validation_ap_weight)\n\n    def _validation_objective(self, y, logits):\n        # standardized partial ROC-AUC by sklearn; the max_fpr is ONLY used\n        # for model selection, NOT a guarantee of future operating-point FPR.\n        ap=average_precision_score(y, logits)\n        if self.low_fpr_validation < 1:\n            pauc=roc_auc_score(y, logits, max_fpr=self.low_fpr_validation)\n        else:\n            pauc=roc_auc_score(y,logits)\n        return self.validation_ap_weight*ap+(1-self.validation_ap_weight)*pauc\n\n    def _weights(self,y,F,iteration):\n        w=np.ones_like(F)\n        if self.hard_negative_strength<=0 or iteration<self.hard_negative_warmup:\n            return w\n        negatives=(y==0)\n        # Train-only order statistic. Never uses validation, calibration, or test.\n        cutoff=np.quantile(F[negatives],1-self.hard_negative_fraction)\n        softness=expit(np.clip((F[negatives]-cutoff)/self.hard_negative_temperature,-30,30))\n        w[negatives]+=self.hard_negative_strength*softness\n        # Average weight normalization does not change Newton ratios, but avoids\n        # exploding regularization interactions during training.\n        return w / w.mean()\n\n    def fit(self,X,y,X_val=None,y_val=None):\n        X=np.asarray(X,dtype=np.float64)\n        y=np.asarray(y)\n        if y.ndim != 1 or len(y) != len(X) or set(np.unique(y)) != {0,1}:\n            raise ValueError(\'y must contain binary labels 0 and 1, one per row\')\n        if X.ndim!=2 or not np.isfinite(X).all():\n            raise ValueError(\'X must be a finite numeric 2D matrix\')\n        if X_val is None:\n            X,X_val,y,y_val=train_test_split(X,y,test_size=.2,stratify=y,random_state=self.random_state)\n        else:\n            X_val=np.asarray(X_val,dtype=np.float64)\n            y_val=np.asarray(y_val)\n        if (X_val.ndim != 2 or X_val.shape[1] != X.shape[1] or\n                y_val.ndim != 1 or len(y_val) != len(X_val)):\n            raise ValueError(\'Validation rows and feature count must match\')\n        if not np.isfinite(X_val).all() or set(np.unique(y))!={0,1} or set(np.unique(y_val))!={0,1}:\n            raise ValueError(\'both classes required and all features must be finite\')\n        # A threshold calibrated for a previous fit does not apply to the new model.\n        for name in list(vars(self)):\n            if name == \'decision_threshold_\' or name.startswith(\'calibration_\'):\n                delattr(self, name)\n        self.scaler_=StandardScaler().fit(X)\n        Xt=self._features(X); Xv=self._features(X_val)\n        self.linear_=LogisticRegression(C=self.linear_C,solver=\'lbfgs\',max_iter=500,\n                                       random_state=self.random_state).fit(Xt,y)\n        F=self.linear_.decision_function(Xt).astype(np.float64)\n        Fv=self.linear_.decision_function(Xv).astype(np.float64)\n        self.steps_=[]\n        self.pairs_=np.array([(i,j) for i in range(X.shape[1])\n                              for j in range(i+1,X.shape[1])],dtype=np.int64).reshape(-1,2)\n        if self.pairs_.size and self.use_cross:\n            ii,jj=self.pairs_[:,0],self.pairs_[:,1]\n            raw=Xt[:,ii]*Xt[:,jj]\n            self.cross_means_=raw.mean(axis=0)\n            self.cross_stds_=np.maximum(raw.std(axis=0),.05)\n            Ct=self._cross_matrix(Xt);Cv=self._cross_matrix(Xv)\n        else:\n            self.cross_means_=np.array([]);self.cross_stds_=np.array([])\n            Ct=Cv=None\n        best=self._validation_objective(y_val,Fv)\n        best_i=0\n        patience=0\n        self.validation_history_=[]\n        for t in range(self.n_estimators):\n            p=expit(np.clip(F,-25,25))\n            weights=self._weights(y,F,t)\n            # Diagonal Newton approximation for a weighted logistic loss.\n            grad=(y-p)*weights\n            hess=np.maximum(p*(1-p),0.002)*weights\n            response=np.clip(grad/hess,-15,15)\n            tree=DecisionTreeRegressor(max_leaf_nodes=self.max_leaf_nodes,\n                min_samples_leaf=self.min_samples_leaf,\n                random_state=self.random_state+t)\n            tree.fit(Xt,response,sample_weight=hess)\n            F+=self.learning_rate*np.clip(tree.predict(Xt),-5,5)\n            Fv+=self.learning_rate*np.clip(tree.predict(Xv),-5,5)\n            cross_info=None\n            if Ct is not None and Ct.shape[1]:\n                p=expit(np.clip(F,-25,25))\n                weights=self._weights(y,F,t)\n                grad=(y-p)*weights\n                hess=np.maximum(p*(1-p),0.002)*weights\n                num=Ct.T@grad\n                den=(Ct**2).T@hess+self.pair_penalty\n                gain=num*num/den\n                j=int(np.argmax(gain))\n                coefficient=np.clip(num[j]/den[j],-3,3)*self.learning_rate*self.pair_rate\n                F+=coefficient*Ct[:,j]\n                Fv+=coefficient*Cv[:,j]\n                cross_info=(j,float(coefficient))\n            self.steps_.append((tree,cross_info))\n            current=self._validation_objective(y_val,Fv)\n            self.validation_history_.append(current)\n            if current > best+0.00001:\n                best=current;best_i=t+1;patience=0\n            else:\n                patience+=1\n            if patience>=self.early_stopping_rounds:break\n        self.best_iteration_=best_i\n        self.validation_objective_=best\n        self.compile()\n        self.validation_logloss_=self._loss(y_val,self.decision_function(X_val))\n        return self\n', 'altcboost_v5.py': '"""Experimental temporal-tail tree boosting with checked empirical descent."""\nimport numpy as np\nfrom scipy.special import expit\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.tree import DecisionTreeRegressor\nfrom detection_objective import selection_key\nfrom tail_objective import tail_objective\n\nclass ALTCBoostV5:\n    def __init__(self,n_estimators=100,learning_rate=.08,max_leaf_nodes=17,\n                 min_samples_leaf=40,leaf_l2=1.,max_leaf_step=5.,tail_fraction=.01,\n                 temperature=1.,group_temperature=.25,anchor_weight=.2,\n                 early_stopping_rounds=16,random_state=17):\n        if (n_estimators<1 or learning_rate<=0 or leaf_l2<0 or max_leaf_step<=0 or\n            max_leaf_nodes<2 or min_samples_leaf<1 or early_stopping_rounds<1):\n            raise ValueError(\'Invalid tree or training settings\')\n        if not (0<tail_fraction<=1 and temperature>0 and group_temperature>0 and 0<=anchor_weight<=1):\n            raise ValueError(\'Invalid objective settings\')\n        self.n_estimators=n_estimators;self.learning_rate=learning_rate\n        self.max_leaf_nodes=max_leaf_nodes;self.min_samples_leaf=min_samples_leaf\n        self.leaf_l2=leaf_l2;self.max_leaf_step=max_leaf_step;self.tail_fraction=tail_fraction\n        self.temperature=temperature;self.group_temperature=group_temperature\n        self.anchor_weight=anchor_weight;self.early_stopping_rounds=early_stopping_rounds\n        self.random_state=random_state\n\n    def objective(self,y,score,groups):\n        return tail_objective(y,score,groups,self.tail_fraction,self.temperature,\n                              self.group_temperature,self.anchor_weight)\n\n    def _features(self,x):\n        x=np.asarray(x,dtype=float)\n        if x.ndim!=2 or x.shape[1]!=self.n_features_ or not np.isfinite(x).all():\n            raise ValueError(\'Expected finite rows with the fitted feature count\')\n        return self.scaler_.transform(x)\n\n    def fit(self,x,y,x_val,y_val,groups=None):\n        x=np.asarray(x,dtype=float);x_val=np.asarray(x_val,dtype=float)\n        y=np.asarray(y);y_val=np.asarray(y_val)\n        if (x.ndim!=2 or x_val.ndim!=2 or x.shape[1]!=x_val.shape[1] or\n            y.ndim!=1 or y_val.ndim!=1 or len(y)!=len(x) or len(y_val)!=len(x_val) or\n            set(np.unique(y))!={0,1} or set(np.unique(y_val))!={0,1} or\n            not np.isfinite(x).all() or not np.isfinite(x_val).all()):\n            raise ValueError(\'Finite matching matrices and both binary classes are required\')\n        groups=np.zeros(len(y),dtype=int) if groups is None else np.asarray(groups)\n        self.n_features_=x.shape[1];self.scaler_=StandardScaler().fit(x)\n        xt=self._features(x);xv=self._features(x_val)\n        self.linear_=LogisticRegression(class_weight=\'balanced\',max_iter=500,\n            random_state=self.random_state).fit(xt,y)\n        score=self.linear_.decision_function(xt);validation=self.linear_.decision_function(xv)\n        value,_,_,_=self.objective(y,score,groups)\n        self.training_loss_=[value];self.steps_=[];self.best_iteration_=0\n        self.validation_history_=[];self.step_diagnostics_=[]\n        best=selection_key(y_val,validation);patience=0\n        for iteration in range(self.n_estimators):\n            value,gradient,hessian,detail=self.objective(y,score,groups)\n            # Scaling preserves the objective direction and sets a per-row L2 scale.\n            gradient=gradient*len(y);hessian=np.maximum(hessian*len(y),1e-6)\n            tree=DecisionTreeRegressor(max_leaf_nodes=self.max_leaf_nodes,\n                min_samples_leaf=self.min_samples_leaf,random_state=self.random_state+iteration)\n            tree.fit(xt,np.clip(-gradient/hessian,-50,50),sample_weight=hessian)\n            leaves=tree.apply(xt)\n            numerator=np.bincount(leaves,weights=-gradient)\n            denominator=np.bincount(leaves,weights=hessian)+self.leaf_l2\n            values=np.clip(numerator/np.maximum(denominator,1e-12),-self.max_leaf_step,self.max_leaf_step)\n            tree.tree_.value[:,0,0]=0.\n            for leaf in np.unique(leaves):tree.tree_.value[leaf,0,0]=values[leaf]\n            direction=tree.predict(xt);accepted=False\n            for backtracks in range(12):\n                rate=self.learning_rate*2.**(-backtracks)\n                candidate=score+rate*direction\n                candidate_loss=self.objective(y,candidate,groups)[0]\n                if candidate_loss<value-1e-12:\n                    accepted=True;break\n            if not accepted:break\n            self.steps_.append((tree,rate));score=candidate\n            validation+=rate*tree.predict(xv)\n            self.training_loss_.append(candidate_loss)\n            self.step_diagnostics_.append(dict(rate=rate,backtracks=backtracks,**detail))\n            key=selection_key(y_val,validation);self.validation_history_.append(key)\n            if key>best:best=key;self.best_iteration_=len(self.steps_);patience=0\n            else:patience+=1\n            if patience>=self.early_stopping_rounds:break\n        self.validation_key_=best\n        return self\n\n    def decision_function(self,x):\n        z=self._features(x);score=self.linear_.decision_function(z)\n        for tree,rate in self.steps_[:self.best_iteration_]:score+=rate*tree.predict(z)\n        return score\n\n    def predict_proba(self,x):\n        # A sigmoid score interface does not establish calibrated fraud probabilities.\n        positive=expit(self.decision_function(x))\n        return np.column_stack([1-positive,positive])\n', 'altcboost_v6.py': '"""Direct pairwise temporal-tail candidate; inherits the checked-descent trainer."""\nfrom altcboost_v5 import ALTCBoostV5\nfrom tail_objective import pairwise_tail_objective\n\nclass ALTCBoostV6(ALTCBoostV5):\n    def objective(self,y,score,groups):\n        return pairwise_tail_objective(y,score,groups,self.tail_fraction,self.temperature,\n                                      self.group_temperature,self.anchor_weight)\n', 'altcboost_v7.py': '"""Research candidate: a revisable linear prior and a small exact-curvature block."""\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.tree import DecisionTreeRegressor\nfrom altcboost_v6 import ALTCBoostV6\nfrom detection_objective import selection_key\nfrom prior_block import directional_derivatives,block_proposals,bounded_prior\n\nclass ALTCBoostV7(ALTCBoostV6):\n    MODES={\'fixed\',\'zero\',\'clipped\',\'shrunk\',\'adaptive_diagonal\',\'adaptive_joint\'}\n    def __init__(self,prior_mode=\'adaptive_joint\',block_damping=.001,**kwargs):\n        super().__init__(**kwargs)\n        if prior_mode not in self.MODES or block_damping<=0:raise ValueError(\'Invalid prior mode or damping\')\n        self.prior_mode=prior_mode;self.block_damping=block_damping\n    def _prior(self,z):\n        if self.linear_ is None:return np.zeros(len(z))\n        score=self.linear_.decision_function(z)\n        return bounded_prior(score) if self.prior_mode==\'clipped\' else score\n    def fit(self,x,y,x_val,y_val,groups=None):\n        x=np.asarray(x,dtype=float);x_val=np.asarray(x_val,dtype=float);y=np.asarray(y);y_val=np.asarray(y_val)\n        if (x.ndim!=2 or x_val.ndim!=2 or x.shape[1]!=x_val.shape[1] or\n            y.ndim!=1 or y_val.ndim!=1 or len(y)!=len(x) or len(y_val)!=len(x_val) or\n            set(np.unique(y))!={0,1} or set(np.unique(y_val))!={0,1} or\n            not np.isfinite(x).all() or not np.isfinite(x_val).all()):\n            raise ValueError(\'Finite matching matrices and both binary classes required\')\n        groups=np.zeros(len(y),dtype=int) if groups is None else np.asarray(groups)\n        self.n_features_=x.shape[1];self.scaler_=StandardScaler().fit(x)\n        xt=self._features(x);xv=self._features(x_val)\n        self.linear_=None if self.prior_mode==\'zero\' else LogisticRegression(\n            class_weight=\'balanced\',max_iter=500,random_state=self.random_state).fit(xt,y)\n        prior=self._prior(xt);prior_val=self._prior(xv)\n        weight=.25 if self.prior_mode==\'shrunk\' else (0. if self.prior_mode==\'zero\' else 1.)\n        score=weight*prior;validation=weight*prior_val\n        self.steps_=[];self.prior_weights_=[weight];self.best_iteration_=0\n        self.training_loss_=[self.objective(y,score,groups)[0]];self.validation_history_=[];self.step_diagnostics_=[]\n        best=selection_key(y_val,validation);patience=0;validation_directions=[]\n        adaptive=self.prior_mode.startswith(\'adaptive\')\n        for iteration in range(self.n_estimators):\n            value,gradient,hessian,_=self.objective(y,score,groups)\n            gradient=gradient*len(y);hessian=np.maximum(hessian*len(y),1e-6)\n            tree=DecisionTreeRegressor(max_leaf_nodes=self.max_leaf_nodes,min_samples_leaf=self.min_samples_leaf,\n                random_state=self.random_state+iteration)\n            tree.fit(xt,np.clip(-gradient/hessian,-50,50),sample_weight=hessian)\n            leaves=tree.apply(xt);numerator=np.bincount(leaves,weights=-gradient)\n            denominator=np.bincount(leaves,weights=hessian)+self.leaf_l2\n            leaf_values=np.clip(numerator/np.maximum(denominator,1e-12),-self.max_leaf_step,self.max_leaf_step)\n            tree.tree_.value[:,0,0]=0.\n            for leaf in np.unique(leaves):tree.tree_.value[leaf,0,0]=leaf_values[leaf]\n            direction=tree.predict(xt);directions=np.column_stack([prior,direction])\n            if adaptive:\n                _,g,h=directional_derivatives(y,score,groups,directions,self.tail_fraction,\n                    self.temperature,self.group_temperature,self.anchor_weight)\n                proposals=block_proposals(g,h,weight,self.learning_rate,self.block_damping,\n                    diagonal=self.prior_mode==\'adaptive_diagonal\')\n            else:proposals=[(\'tree_only\',np.array([0.,self.learning_rate]))]\n            accepted=None\n            for kind,proposal in proposals:\n                if np.all(proposal==0):continue\n                for backtracks in range(12):\n                    step=proposal*2.**(-backtracks);candidate=score+directions@step\n                    loss=self.objective(y,candidate,groups)[0]\n                    if loss<value-1e-12:\n                        if accepted is None or loss<accepted[\'loss\']:\n                            accepted=dict(kind=kind,step=step,score=candidate,loss=loss,backtracks=backtracks)\n                        break\n            if accepted is None:break\n            da,rate=accepted[\'step\'];weight=float(np.clip(weight+da,0,1))\n            score=accepted[\'score\']\n            self.steps_.append((tree,float(rate)));self.prior_weights_.append(weight)\n            validation_directions.append(tree.predict(xv))\n            # Rebuild the selected representation when the prior coefficient changes.\n            validation=weight*prior_val\n            for (_,coefficient),prediction in zip(self.steps_,validation_directions):validation+=coefficient*prediction\n            self.training_loss_.append(accepted[\'loss\'])\n            self.step_diagnostics_.append(dict(kind=accepted[\'kind\'],prior_weight=weight,prior_delta=float(da),tree_rate=float(rate),backtracks=accepted[\'backtracks\']))\n            key=selection_key(y_val,validation);self.validation_history_.append(key)\n            if key>best:best=key;self.best_iteration_=len(self.steps_);patience=0\n            else:patience+=1\n            if patience>=self.early_stopping_rounds:break\n        self.validation_key_=best\n        return self\n    def decision_function(self,x):\n        z=self._features(x);score=self.prior_weights_[self.best_iteration_]*self._prior(z)\n        for tree,rate in self.steps_[:self.best_iteration_]:\n            if rate:score+=rate*tree.predict(z)\n        return score\n', 'evaluation.py': '"""Shared calibration and held-out metrics for all benchmark families."""\nimport numpy as np\nfrom scipy.stats import beta\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\ndef calibrate(y, score, alpha=.001, confidence=.95):\n    y, score = np.asarray(y), np.asarray(score, dtype=float)\n    if y.ndim != 1 or score.shape != y.shape or not np.isfinite(score).all():\n        raise ValueError(\'Expected equally sized finite one-dimensional arrays\')\n    if not set(np.unique(y)).issubset({0, 1}) or not 0 < alpha < 1 or not 0 < confidence < 1:\n        raise ValueError(\'Invalid labels or calibration settings\')\n    neg = np.sort(score[y == 0])[::-1]\n    if not len(neg):\n        raise ValueError(\'Calibration requires negative examples\')\n    # This bound assumes independent calibration negatives with the future negative distribution.\n    k = np.arange(int(np.floor(alpha * len(neg))) + 1)\n    upper = beta.ppf(confidence, k + 1, len(neg) - k)\n    feasible = k[upper <= alpha]\n    if not len(feasible):\n        raise ValueError(\'Insufficient calibration negatives for requested confidence\')\n    allowed = int(feasible[-1])\n    return float(np.nextafter(neg[allowed], np.inf)), dict(\n        negative_count=len(neg), allowed=allowed,\n        tolerance_bound=float(upper[allowed]), alpha=alpha, confidence=confidence)\n\ndef metrics(y, score, threshold, alpha=.001):\n    # A float64 nextafter cutoff can round back to a tie in float32 comparisons.\n    y, score = np.asarray(y), np.asarray(score,dtype=np.float64)\n    flags = score >= threshold\n    pos = y == 1; neg = y == 0\n    tp = int((flags & pos).sum()); fp = int((flags & neg).sum())\n    p, n = int(pos.sum()), int(neg.sum())\n    if not p or not n:\n        raise ValueError(\'Test metrics require both classes\')\n    return dict(ap=float(average_precision_score(y, score)),\n                pauc=float(roc_auc_score(y, score, max_fpr=alpha)),\n                tp=tp, fp=fp, fn=p-tp, tn=n-fp, recall=tp/p, fpr=fp/n,\n                recall_lower_95=float(beta.ppf(.025,tp,p-tp+1)) if tp else 0.,\n                recall_upper_95=float(beta.ppf(.975,tp+1,p-tp)) if tp < p else 1.,\n                fpr_upper_95=float(beta.ppf(.95, fp+1, n-fp)) if fp < n else 1.,\n                precision=tp/max(tp+fp, 1), alerts=tp+fp)\n', 'detection_objective.py': '"""Development selection at fixed empirical false-positive budgets."""\nimport numpy as np\nfrom sklearn.metrics import average_precision_score\n\ndef empirical_cutoff(y,score,alpha):\n    y,score=np.asarray(y),np.asarray(score,dtype=float)\n    if y.ndim!=1 or score.shape!=y.shape or not np.isfinite(score).all():\n        raise ValueError(\'Expected equally sized finite label and score arrays\')\n    if set(np.unique(y))!={0,1} or not 0<alpha<1:\n        raise ValueError(\'Both binary classes and a valid budget are required\')\n    negative=np.sort(score[y==0])[::-1]\n    return float(np.nextafter(negative[int(np.floor(alpha*len(negative)))],np.inf))\n\ndef selection_key(y,score,budgets=(.0005,.001)):\n    # Float64 comparisons preserve a nextafter cutoff above tied float32 scores.\n    y=np.asarray(y);score=np.asarray(score,dtype=np.float64)\n    recall=[];false_positive=[]\n    for alpha in budgets:\n        flags=score>=empirical_cutoff(y,score,alpha)\n        recall.append(float(flags[y==1].mean()))\n        false_positive.append(float(flags[y==0].mean()))\n    # This is a validation ranking rule, not a constraint on future traffic.\n    return (float(np.mean(recall)),-float(np.mean(false_positive)),\n            float(average_precision_score(y,score)))\n', 'tail_objective.py': '"""Temporal negative-tail ranking surrogate with exact diagonal derivatives.\n\nThis combines established entropic ranking and group-robust ideas. It is not\nan exact partial-AUC objective or a bound on future false-positive rates.\n"""\nimport numpy as np\nfrom scipy.special import expit,logsumexp\n\ndef temporal_groups(times,count=4):\n    times=np.asarray(times)\n    if times.ndim!=1 or not len(times) or not np.isfinite(times).all() or count<1:\n        raise ValueError(\'Finite training times and a positive group count are required\')\n    unique=np.unique(times)\n    if len(unique)<count:raise ValueError(\'Too few distinct training times\')\n    edges=unique[[int(len(unique)*i/count) for i in range(1,count)]]\n    return np.searchsorted(edges,times,side=\'right\'),edges\n\ndef evaluation_windows(times):\n    times=np.asarray(times)\n    # Coarse source timestamps cannot support invented sub-periods.\n    return temporal_groups(times,count=min(4,len(np.unique(times))))\n\ndef tail_objective(y,score,groups,tail_fraction=.01,temperature=1.,\n                   group_temperature=.25,anchor_weight=.2,margin=1.):\n    y=np.asarray(y);score=np.asarray(score,dtype=float);groups=np.asarray(groups)\n    if (y.ndim!=1 or score.shape!=y.shape or groups.shape!=y.shape or\n        set(np.unique(y))!={0,1} or not np.isfinite(score).all() or\n        not np.isfinite(groups).all()):\n        raise ValueError(\'Finite matching scores, groups and both binary classes are required\')\n    if not (0<tail_fraction<=1 and temperature>0 and group_temperature>0 and 0<=anchor_weight<=1):\n        raise ValueError(\'Invalid objective settings\')\n    positive=y==1;negative=~positive\n    weights=np.where(positive,.5/positive.sum(),.5/negative.sum())\n    probability=expit(score)\n    anchor=float(np.sum(weights*np.logaddexp(0,np.where(positive,-score,score))))\n    gradient=anchor_weight*weights*(probability-y)\n    hessian=anchor_weight*weights*probability*(1-probability)\n    if anchor_weight==1:return anchor,gradient,hessian,dict(group_losses=[],group_weights=[])\n    blocks=[]\n    for group in np.unique(groups):\n        pos=np.flatnonzero((groups==group)&positive)\n        neg=np.flatnonzero((groups==group)&negative)\n        if not len(pos) or not len(neg):raise ValueError(\'Both classes are required in every training group\')\n        count=max(1,int(np.ceil(tail_fraction*len(neg))))\n        # The loss is nonsmooth at a negative-tail membership tie.\n        neg=neg[np.argsort(-score[neg],kind=\'stable\')[:count]]\n        a=score[neg]/temperature;b=-score[pos]/temperature\n        la=logsumexp(a);lb=logsumexp(b)\n        z=la-np.log(len(neg))+lb-np.log(len(pos))+margin/temperature\n        loss=float(np.logaddexp(0,z));d=expit(z)\n        qn=np.exp(a-la);qp=np.exp(b-lb)\n        gn=d*qn/temperature;gp=-d*qp/temperature\n        hn=d*qn*(1-d*qn)/temperature**2\n        hp=d*qp*(1-d*qp)/temperature**2\n        blocks.append((loss,neg,pos,gn,gp,hn,hp))\n    losses=np.array([block[0] for block in blocks])\n    scaled=losses/group_temperature\n    robust=float(group_temperature*(logsumexp(scaled)-np.log(len(blocks))))\n    block_weights=np.exp(scaled-logsumexp(scaled))\n    for r,(_,neg,pos,gn,gp,hn,hp) in zip(block_weights,blocks):\n        for index,g,h in [(neg,gn,hn),(pos,gp,hp)]:\n            gradient[index]+=(1-anchor_weight)*r*g\n            hessian[index]+=(1-anchor_weight)*(r*h+r*(1-r)*g*g/group_temperature)\n    value=anchor_weight*anchor+(1-anchor_weight)*robust\n    return float(value),gradient,hessian,dict(group_losses=losses.tolist(),group_weights=block_weights.tolist())\n\ndef pairwise_tail_objective(y,score,groups,tail_fraction=.01,temperature=1.,\n                            group_temperature=.25,anchor_weight=.2,margin=1.,chunk_size=64):\n    """Exact mean logistic pair loss over the selected negative tail, in chunks.\n\n    This is a partial-AUC surrogate, not the discontinuous partial AUC itself.\n    Chunking bounds memory; it does not remove the pairwise arithmetic cost.\n    """\n    anchor,ag,ah,_=tail_objective(y,score,groups,tail_fraction,temperature,\n                                 group_temperature,1.,margin)\n    if not 0<=anchor_weight<=1 or chunk_size<1:raise ValueError(\'Invalid pairwise objective settings\')\n    if anchor_weight==1:return anchor,ag,ah,dict(group_losses=[],group_weights=[])\n    y=np.asarray(y);score=np.asarray(score,dtype=float);groups=np.asarray(groups)\n    gradient=anchor_weight*ag;hessian=anchor_weight*ah;blocks=[]\n    for group in np.unique(groups):\n        pos=np.flatnonzero((groups==group)&(y==1));neg=np.flatnonzero((groups==group)&(y==0))\n        if not len(pos) or not len(neg):raise ValueError(\'Both classes are required in every training group\')\n        count=max(1,int(np.ceil(tail_fraction*len(neg))))\n        neg=neg[np.argsort(-score[neg],kind=\'stable\')[:count]]\n        gn=np.zeros(len(neg));hn=np.zeros(len(neg));gp=np.zeros(len(pos));hp=np.zeros(len(pos));total=0.\n        pairs=len(pos)*len(neg)\n        for start in range(0,len(pos),chunk_size):\n            stop=min(start+chunk_size,len(pos))\n            difference=(score[neg][None,:]-score[pos[start:stop]][:,None]+margin)/temperature\n            probability=expit(difference);curvature=probability*(1-probability)\n            total+=float(np.logaddexp(0,difference).sum())/pairs\n            gn+=probability.sum(axis=0)/(pairs*temperature)\n            gp[start:stop]=-probability.sum(axis=1)/(pairs*temperature)\n            hn+=curvature.sum(axis=0)/(pairs*temperature**2)\n            hp[start:stop]=curvature.sum(axis=1)/(pairs*temperature**2)\n        blocks.append((total,neg,pos,gn,gp,hn,hp))\n    losses=np.array([block[0] for block in blocks]);scaled=losses/group_temperature\n    robust=float(group_temperature*(logsumexp(scaled)-np.log(len(blocks))))\n    block_weights=np.exp(scaled-logsumexp(scaled))\n    for r,(_,neg,pos,gn,gp,hn,hp) in zip(block_weights,blocks):\n        for index,g,h in [(neg,gn,hn),(pos,gp,hp)]:\n            gradient[index]+=(1-anchor_weight)*r*g\n            hessian[index]+=(1-anchor_weight)*(r*h+r*(1-r)*g*g/group_temperature)\n    return anchor_weight*anchor+(1-anchor_weight)*robust,gradient,hessian,dict(\n        group_losses=losses.tolist(),group_weights=block_weights.tolist())\n', 'prior_block.py': '"""Exact small directional Hessian for a temporal negative-tail pair loss."""\nimport numpy as np\nfrom scipy.special import expit\nfrom tail_objective import pairwise_tail_objective\n\ndef bounded_prior(score,limit=4.):\n    if limit<=0:raise ValueError(\'Positive prior bound required\')\n    return limit*np.tanh(np.asarray(score,dtype=float)/limit)\n\ndef directional_derivatives(y,score,groups,directions,tail_fraction=.01,temperature=1.,\n                            group_temperature=.25,anchor_weight=.2,chunk_size=64):\n    value,row_gradient,_,detail=pairwise_tail_objective(y,score,groups,tail_fraction,\n        temperature,group_temperature,anchor_weight,chunk_size=chunk_size)\n    y=np.asarray(y);score=np.asarray(score,dtype=float);groups=np.asarray(groups)\n    directions=np.asarray(directions,dtype=float)\n    if directions.ndim!=2 or directions.shape[0]!=len(y) or not np.isfinite(directions).all():\n        raise ValueError(\'Finite directions with one row per example required\')\n    gradient=directions.T@row_gradient\n    weights=np.where(y==1,.5/np.sum(y==1),.5/np.sum(y==0))\n    p=expit(score);hessian=anchor_weight*(directions.T@(directions*(weights*p*(1-p))[:,None]))\n    if anchor_weight==1:return value,gradient,hessian\n    block_gradients=[];block_hessians=[]\n    for group in np.unique(groups):\n        pos=np.flatnonzero((groups==group)&(y==1));neg=np.flatnonzero((groups==group)&(y==0))\n        count=max(1,int(np.ceil(tail_fraction*len(neg))))\n        neg=neg[np.argsort(-score[neg],kind=\'stable\')[:count]]\n        pairs=len(pos)*len(neg);g=np.zeros(directions.shape[1]);h=np.zeros((len(g),len(g)))\n        for start in range(0,len(pos),chunk_size):\n            selected=pos[start:start+chunk_size]\n            delta=(score[neg][None,:]-score[selected][:,None]+1.)/temperature\n            probability=expit(delta).ravel();curvature=probability*(1-probability)\n            difference=((directions[neg][None,:,:]-directions[selected][:,None,:])/temperature).reshape(-1,len(g))\n            g+=difference.T@probability/pairs\n            h+=difference.T@(difference*curvature[:,None])/pairs\n        block_gradients.append(g);block_hessians.append(h)\n    r=np.asarray(detail[\'group_weights\']);bg=np.asarray(block_gradients);mean=r@bg\n    robust_hessian=sum(weight*h for weight,h in zip(r,block_hessians))\n    robust_hessian+=(bg.T@(bg*r[:,None])-np.outer(mean,mean))/group_temperature\n    hessian+=(1-anchor_weight)*robust_hessian\n    return value,gradient,(hessian+hessian.T)/2\n\ndef block_proposals(gradient,hessian,prior_weight,learning_rate=.08,damping=.001,diagonal=False):\n    g=np.asarray(gradient,dtype=float);h=np.asarray(hessian,dtype=float)\n    if g.shape!=(2,) or h.shape!=(2,2) or not np.isfinite(g).all() or not np.isfinite(h).all():\n        raise ValueError(\'Finite two-coordinate derivatives required\')\n    if not 0<=prior_weight<=1 or learning_rate<=0 or damping<=0:raise ValueError(\'Invalid block constraints\')\n    scale=np.maximum(np.sqrt(np.maximum(np.diag(h),0)),1e-6)\n    normalized=h/np.outer(scale,scale)\n    if diagonal:normalized=np.diag(np.diag(normalized))\n    step=-np.linalg.solve(normalized+damping*np.eye(2),g/scale)/scale\n    joint=np.array([np.clip(step[0],-prior_weight,1-prior_weight),np.clip(step[1],0,learning_rate)])\n    prior=np.array([np.clip(-g[0]/max(h[0,0]*(1+damping),1e-12),-prior_weight,1-prior_weight),0.])\n    return [(\'tree_only\',np.array([0.,learning_rate])),(\'prior_only\',prior),(\'joint\',joint)]\n', 'multi_data.py': '"""Read-only discovery and deterministic cohorts for three synthetic fraud sources."""\nimport argparse\nimport hashlib\nimport json\nfrom datetime import datetime,timezone\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nBAF_NUM=[\'income\',\'name_email_similarity\',\'prev_address_months_count\',\'current_address_months_count\',\n \'customer_age\',\'days_since_request\',\'intended_balcon_amount\',\'zip_count_4w\',\'velocity_6h\',\n \'velocity_24h\',\'velocity_4w\',\'bank_branch_count_8w\',\'date_of_birth_distinct_emails_4w\',\n \'credit_risk_score\',\'email_is_free\',\'phone_home_valid\',\'phone_mobile_valid\',\'bank_months_count\',\n \'has_other_cards\',\'proposed_credit_limit\',\'foreign_request\',\'session_length_in_minutes\',\n \'keep_alive_session\',\'device_distinct_emails_8w\']\nSPECS={\n \'baf_base\':dict(ref=\'sgpjesus/bank-account-fraud-dataset-neurips-2022\',files=[\'Base.csv\'],\n  label=\'fraud_bool\',time=\'month\',numeric=BAF_NUM,\n  categorical=[\'payment_type\',\'employment_status\',\'housing_status\',\'source\',\'device_os\'],group=None,\n  kind=\'synthetic, generated from anonymized real application data\',unit=\'bank-account application row\',\n  licence=\'CC-BY-NC-SA-4.0\',source=\'https://github.com/Feedzai/bank-account-fraud\',\n  excluded_reason=\'month is used only for partitioning; device_fraud_count is excluded as a fraud-history field\'),\n \'paysim\':dict(ref=\'ealaxi/paysim1\',files=[\'PS_20174392719_1491204439457_log.csv\'],\n  label=\'isFraud\',time=\'step\',numeric=[\'amount\',\'oldbalanceOrg\',\'oldbalanceDest\'],\n  categorical=[\'type\'],group=\'nameOrig\',kind=\'synthetic mobile-money transactions\',unit=\'transaction row\',\n  licence=\'CC-BY-SA-4.0\',source=\'https://github.com/EdgarLopezPhD/PaySim\',\n  excluded_reason=\'newbalanceOrig/newbalanceDest are post-event balances; isFlaggedFraud is another rule decision; names are not predictors\'),\n \'sparkov\':dict(ref=\'kartik2112/fraud-detection\',files=[\'fraudTrain.csv\',\'fraudTest.csv\'],\n  label=\'is_fraud\',time=\'trans_date_trans_time\',numeric=[\'amt\',\'city_pop\'],\n  categorical=[\'category\'],group=\'cc_num\',kind=\'synthetic card transactions generated with Sparkov\',unit=\'card-transaction row\',\n  licence=\'CC0-1.0\',source=\'https://github.com/namebrandon/Sparkov_Data_Generation\',\n  excluded_reason=\'IDs, names, addresses, coordinates, birth dates and merchant identities are excluded; card ID is used only for overlap auditing\')}\nPARTS=[\'train\',\'validation\',\'calibration\',\'test\']\n\ndef feature_names(key):\n    spec=SPECS[key]\n    return spec[\'numeric\']+([\'hour_of_day\',\'day_of_week\'] if key==\'sparkov\' else [])+spec[\'categorical\']\n\ndef read_columns(key):\n    spec=SPECS[key]\n    return list(dict.fromkeys([spec[\'label\'],spec[\'time\'],*spec[\'numeric\'],*spec[\'categorical\'],\n                              *([spec[\'group\']] if spec[\'group\'] else [])]))\n\ndef read_chunks(key,paths,chunksize=100000):\n    spec=SPECS[key];offset=0\n    for file_number,path in enumerate(paths):\n        dtype={column:\'str\' for column in [*spec[\'categorical\'],*([spec[\'group\']] if spec[\'group\'] else [])]}\n        for frame in pd.read_csv(path,usecols=read_columns(key),dtype=dtype,chunksize=chunksize):\n            if not set(frame[spec[\'label\']].dropna().unique()).issubset({0,1}) or frame[spec[\'label\']].isna().any():\n                raise ValueError(\'Source labels must be present and binary\')\n            if key==\'sparkov\':\n                stamp=pd.to_datetime(frame[spec[\'time\']],utc=True,errors=\'raise\').dt.as_unit(\'ns\')\n                if stamp.isna().any():raise ValueError(\'Source time must be present\')\n                frame[\'__time\']=stamp.astype(\'int64\')\n                frame[\'hour_of_day\']=stamp.dt.hour.astype(float)\n                frame[\'day_of_week\']=stamp.dt.day_of_week.astype(float)\n            else:\n                frame[\'__time\']=pd.to_numeric(frame[spec[\'time\']],errors=\'raise\')\n            if not np.isfinite(frame[\'__time\'].to_numpy(dtype=float)).all():\n                raise ValueError(\'Source time must be finite\')\n            for column in spec[\'numeric\']:\n                frame[column]=pd.to_numeric(frame[column],errors=\'raise\').astype(float)\n                if np.isinf(frame[column].to_numpy()).any():raise ValueError(\'Infinite predictor values are unsupported\')\n            for column in spec[\'categorical\']:\n                frame[column]=frame[column].astype(object).where(frame[column].notna(),np.nan)\n            frame[\'__row\']=np.arange(offset,offset+len(frame),dtype=np.int64)\n            offset+=len(frame)\n            frame[\'__file\']=file_number\n            yield frame\n\ndef file_sha256(path):\n    value=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda:stream.read(8*1024*1024),b\'\'):value.update(block)\n    return value.hexdigest()\n\ndef priorities(row_ids,seed):\n    # Stable row-only priorities allow label-blind sampling independent of CSV chunk size.\n    with np.errstate(over=\'ignore\'):\n        value=np.asarray(row_ids,dtype=np.uint64)+np.uint64(seed)\n        value=(value^(value>>np.uint64(30)))*np.uint64(0xbf58476d1ce4e5b9)\n        value=(value^(value>>np.uint64(27)))*np.uint64(0x94d049bb133111eb)\n        return value^(value>>np.uint64(31))\n\ndef partition_ids(key,frame,edges):\n    if key==\'sparkov\':\n        ids=np.searchsorted(edges,frame[\'__time\'].to_numpy(),side=\'right\')\n        ids[frame[\'__file\'].to_numpy()==1]=3\n        return ids\n    return np.searchsorted(edges,frame[\'__time\'].to_numpy(),side=\'right\')\n\ndef load_cohort(key,paths,train_cap=150000,eval_cap=100000,seed=20261112,chunksize=100000):\n    if key not in SPECS:raise ValueError(\'Unknown dataset\')\n    if train_cap<1 or eval_cap<1:raise ValueError(\'Cohort caps must be positive\')\n    spec=SPECS[key];paths=list(map(Path,paths))\n    if len(paths)!=len(spec[\'files\']):raise ValueError(\'Expected all source files in the declared order\')\n    times=[];files=[dict(name=path.name,sha256=file_sha256(path),rows=0,positive=0,\n                       time_min=None,time_max=None) for path in paths]\n    for chunk in read_chunks(key,paths,chunksize):\n        index=int(chunk[\'__file\'].iloc[0]);info=files[index]\n        info[\'rows\']+=len(chunk);info[\'positive\']+=int(chunk[spec[\'label\']].sum())\n        low,high=int(chunk.__time.min()),int(chunk.__time.max())\n        info[\'time_min\']=low if info[\'time_min\'] is None else min(info[\'time_min\'],low)\n        info[\'time_max\']=high if info[\'time_max\'] is None else max(info[\'time_max\'],high)\n        if key!=\'sparkov\' or index==0:times.append(np.unique(chunk[\'__time\']))\n    unique=np.unique(np.concatenate(times))\n    fractions=(.70,.85) if key==\'sparkov\' else (.55,.70,.85)\n    if len(unique)<8:raise ValueError(\'Insufficient distinct time groups for this protocol\')\n    edges=np.array([unique[int(len(unique)*q)] for q in fractions])\n    if key==\'sparkov\' and files[0][\'time_max\']>=files[1][\'time_min\']:\n        raise ValueError(\'Official card training and test time windows overlap\')\n    selected=[None]*4\n    totals=[dict(rows=0,positive=0) for _ in PARTS]\n    missing={column:0 for column in feature_names(key)}\n    for chunk in read_chunks(key,paths,chunksize):\n        chunk[\'__priority\']=priorities(chunk[\'__row\'],seed)\n        ids=partition_ids(key,chunk,edges)\n        for column in missing:missing[column]+=int(chunk[column].isna().sum())\n        for index,name in enumerate(PARTS):\n            part=chunk.loc[ids==index]\n            totals[index][\'rows\']+=len(part);totals[index][\'positive\']+=int(part[spec[\'label\']].sum())\n            if not len(part):continue\n            capacity=train_cap if index==0 else eval_cap\n            combined=part if selected[index] is None else pd.concat([selected[index],part],ignore_index=True)\n            selected[index]=combined.nsmallest(capacity,[\'__priority\',\'__row\']).copy()\n    reports={};frames=[]\n    for index,name in enumerate(PARTS):\n        frame=selected[index]\n        if frame is None or set(frame[spec[\'label\']].unique())!={0,1}:\n            raise ValueError(f\'Both classes are required in sampled partition {name}\')\n        frame=frame.sort_values(\'__row\').reset_index(drop=True);frames.append(frame)\n        reports[name]=dict(source_rows=totals[index][\'rows\'],source_positive=totals[index][\'positive\'],\n            sampled_rows=len(frame),sampled_positive=int(frame[spec[\'label\']].sum()),\n            sampled_negative=int((frame[spec[\'label\']]==0).sum()),\n            time_min=int(frame.__time.min()),time_max=int(frame.__time.max()),\n            cohort_sha256=hashlib.sha256(frame[\'__row\'].to_numpy(dtype=np.int64).tobytes()).hexdigest(),\n            equivalent_predictor_label_rows=int(frame.duplicated(subset=[spec[\'label\'],*feature_names(key)]).sum()))\n    for first,second in zip(frames,frames[1:]):\n        if first.__time.max()>=second.__time.min():raise ValueError(\'Sampled temporal partitions overlap\')\n    entity=dict(status=\'unavailable; source has no applicant identifier\')\n    if spec[\'group\']:\n        train_groups=set(frames[0][spec[\'group\']].dropna().astype(str))\n        test_groups=set(frames[3][spec[\'group\']].dropna().astype(str))\n        entity=dict(scope=\'sampled rows only\',train_groups=len(train_groups),test_groups=len(test_groups),\n                    overlapping_groups=len(train_groups&test_groups),ids_used_as_predictors=False)\n    report=dict(dataset=key,dataset_ref=spec[\'ref\'],dataset_kind=spec[\'kind\'],unit=spec[\'unit\'],\n        source_url=spec[\'source\'],licence=spec[\'licence\'],observed_utc=datetime.now(timezone.utc).isoformat(),\n        files=files,source_rows=sum(info[\'rows\'] for info in files),partitions=reports,\n        time_edges=edges.tolist(),time_unit=\'nanoseconds since Unix epoch\' if key==\'sparkov\' else spec[\'time\'],\n        raw_read_column_allowlist=read_columns(key),predictor_allowlist=feature_names(key),\n        excluded_reason=spec[\'excluded_reason\'],source_missing_predictor_values=missing,entity_overlap=entity,\n        scope=\'Full-source metadata with capped, deterministic, label-blind temporal cohorts; no row export\',\n        sampling=dict(seed=seed,train_cap=train_cap,eval_cap=eval_cap,method=\'lowest stable hash priorities within each time partition\'),\n        duplicate_policy=\'Equal model-input/label rows are counted but retained; equality does not prove a duplicate transaction\',\n        hypotheses=[\'All four sampled time partitions contain both classes\',\n                    \'No time group crosses partition boundaries\',\n                    \'Identifiers, labels and specified post-event fields are absent from predictors\',\n                    \'Identical source files and sampling seed reproduce the cohort hashes\'])\n    return frames,report\n\ndef find_sources(key,root):\n    root=Path(root);result=[]\n    for name in SPECS[key][\'files\']:\n        matches=list(root.rglob(name))\n        if len(matches)!=1:raise ValueError(f\'Expected exactly one source file named {name}\')\n        result.append(matches[0])\n    return result\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'dataset\',choices=SPECS);parser.add_argument(\'root\')\n    parser.add_argument(\'output\');parser.add_argument(\'--train-cap\',type=int,default=150000)\n    parser.add_argument(\'--eval-cap\',type=int,default=100000)\n    args=parser.parse_args();_,report=load_cohort(args.dataset,find_sources(args.dataset,args.root),args.train_cap,args.eval_cap)\n    Path(args.output).parent.mkdir(parents=True,exist_ok=True)\n    Path(args.output).write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n', 'multi_benchmark.py': '"""Frozen-configuration evaluation across separate synthetic fraud sources."""\nimport argparse\nimport importlib.metadata\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.feature_selection import VarianceThreshold\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.pipeline import Pipeline,make_pipeline\nfrom sklearn.preprocessing import OneHotEncoder,StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom threadpoolctl import threadpool_limits\nfrom evaluation import calibrate,metrics\nfrom multi_data import SPECS,PARTS,feature_names,find_sources,load_cohort\n\ndef preprocessing(key):\n    spec=SPECS[key]\n    numeric=spec[\'numeric\']+([\'hour_of_day\',\'day_of_week\'] if key==\'sparkov\' else [])\n    transformer=ColumnTransformer([\n        (\'numeric\',SimpleImputer(strategy=\'median\',add_indicator=True,keep_empty_features=True),numeric),\n        (\'category\',Pipeline([\n            (\'missing\',SimpleImputer(strategy=\'constant\',fill_value=\'missing\')),\n            (\'encode\',OneHotEncoder(handle_unknown=\'infrequent_if_exist\',max_categories=32,\n                                   min_frequency=100,sparse_output=False,dtype=np.float64))]),spec[\'categorical\'])\n    ],remainder=\'drop\',sparse_threshold=0.)\n    return Pipeline([(\'prepare\',transformer),(\'remove_train_constants\',VarianceThreshold())])\n\ndef frozen_models():\n    from altcboost_v3 import ALTCBoostV3\n    from catboost import CatBoostClassifier\n    from xgboost import XGBClassifier\n    yield \'ALTCBoost_incumbent\',dict(trees=70,leaves=9,hard_negative_strength=2.5),ALTCBoostV3(\n        n_estimators=70,max_leaf_nodes=9,min_samples_leaf=40,use_cross=False,random_state=17)\n    yield \'ALTCBoost_tuned\',dict(trees=100,leaves=17,hard_negative_strength=2.5,validation_max_fpr=.001),ALTCBoostV3(\n        n_estimators=100,max_leaf_nodes=17,min_samples_leaf=40,use_cross=False,\n        low_fpr_validation=.001,hard_negative_strength=2.5,random_state=17)\n    yield \'CatBoost\',dict(trees=100,depth=6,positive_weight=5.),CatBoostClassifier(\n        iterations=100,depth=6,scale_pos_weight=5.,learning_rate=.06,random_seed=17,\n        thread_count=2,verbose=False,allow_writing_files=False)\n    yield \'XGBoost\',dict(trees=100,depth=7,positive_weight=10.),XGBClassifier(\n        n_estimators=100,max_depth=7,scale_pos_weight=10.,learning_rate=.06,\n        tree_method=\'hist\',random_state=17,n_jobs=2)\n    yield \'Logistic\',dict(max_iter=500),make_pipeline(StandardScaler(),LogisticRegression(max_iter=500))\n\ndef run(key,paths,output,train_cap=150000,eval_cap=100000,models=None,chunksize=100000):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    frames,discovery=load_cohort(key,paths,train_cap,eval_cap,chunksize=chunksize)\n    (output/\'discovery.json\').write_text(json.dumps(discovery,indent=2))\n    names=feature_names(key);label=SPECS[key][\'label\']\n    processor=preprocessing(key)\n    matrices=[processor.fit_transform(frames[0][names])]\n    matrices.extend(processor.transform(frame[names]) for frame in frames[1:])\n    matrices=[np.ascontiguousarray(matrix,dtype=np.float64) for matrix in matrices]\n    if any(not np.isfinite(matrix).all() for matrix in matrices):raise ValueError(\'Preprocessing produced nonfinite features\')\n    labels=[frame[label].to_numpy(dtype=np.int64) for frame in frames]\n    xtrain,xval,xcal,xtest=matrices;ytrain,yval,ycal,ytest=labels\n    (output/\'preprocessing.json\').write_text(json.dumps(dict(fitted_on=\'sampled training rows only\',\n        predictor_allowlist=names,encoded_features=xtrain.shape[1],\n        common_pipeline=\'training-median numeric imputation plus indicators; bounded one-hot categories; drop training constants\',\n        native_categorical_baselines=False),indent=2))\n    rows=[];configuration=[]\n    for method,config,model in (frozen_models() if models is None else models):\n        started=time.perf_counter()\n        with threadpool_limits(limits=2):\n            if method.startswith(\'ALTCBoost\'):model.fit(xtrain,ytrain,xval,yval)\n            else:model.fit(xtrain,ytrain)\n        duration=time.perf_counter()-started\n        pc=np.asarray(model.predict_proba(xcal)[:,1],dtype=np.float64)\n        pe=np.asarray(model.predict_proba(xtest)[:,1],dtype=np.float64)\n        configuration.append(dict(method=method,config=config,fit_s=duration,\n            selected_iteration=getattr(model,\'best_iteration_\',None)))\n        for alpha in [.0005,.001,.01]:\n            try:threshold,details=calibrate(ycal,pc,alpha)\n            except ValueError as error:\n                if \'Insufficient calibration negatives\' not in str(error):raise\n                rows.append(dict(dataset=key,method=method,alpha=alpha,status=\'unsupported\',reason=str(error)))\n                continue\n            row=dict(dataset=key,dataset_kind=SPECS[key][\'kind\'],unit=SPECS[key][\'unit\'],\n                method=method,status=\'evaluated\',fit_s=duration,threshold=threshold,\n                **details,**metrics(ytest,pe,threshold,alpha))\n            row[\'test_positive\']=int((ytest==1).sum());row[\'test_negative\']=int((ytest==0).sum())\n            rows.append(row)\n            print(key,method,alpha,\'TP\',row[\'tp\'],\'FP\',row[\'fp\'],\'recall\',round(row[\'recall\'],4),flush=True)\n        pd.DataFrame(rows).to_csv(output/\'results.csv\',index=False)\n        (output/\'models.json\').write_text(json.dumps(configuration,indent=2))\n    manifest=dict(dataset=key,source_ref=SPECS[key][\'ref\'],dataset_kind=SPECS[key][\'kind\'],\n        source_url=SPECS[key][\'source\'],licence=SPECS[key][\'licence\'],unit=SPECS[key][\'unit\'],\n        scope=\'New synthetic-dataset evaluation using frozen ULB-selected hyperparameters; no per-dataset model search\',\n        common_cohort=True,common_preprocessing=True,train_cap=train_cap,eval_cap=eval_cap,\n        budgets=[.0005,.001,.01],model_count=len(configuration),\n        limitations=\'Capped label-blind cohorts, not full-data training. Same transformed features for all models; native categorical optimizations not assessed. ALTC uses validation early stopping; other models use fixed tree counts. Sampling seeds are not independent datasets. Temporal and entity dependence can violate calibration assumptions. No production or real-banking validation claim.\',\n        versions={p:importlib.metadata.version(p) for p in [\'numpy\',\'pandas\',\'scipy\',\'scikit-learn\',\'numba\',\'xgboost\',\'catboost\']})\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    return rows,discovery\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'dataset\',choices=SPECS);parser.add_argument(\'root\')\n    parser.add_argument(\'output\');parser.add_argument(\'--train-cap\',type=int,default=150000)\n    parser.add_argument(\'--eval-cap\',type=int,default=100000);args=parser.parse_args()\n    run(args.dataset,find_sources(args.dataset,args.root),args.output,args.train_cap,args.eval_cap)\n', 'validation_data.py': '"""Shared development cohorts and a separately labelled real Bitcoin AML transfer test."""\nimport argparse\nimport hashlib\nimport json\nfrom datetime import datetime,timezone\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.feature_selection import VarianceThreshold\nfrom multi_data import SPECS as ORIGINAL,PARTS,load_cohort,find_sources,feature_names,file_sha256,priorities\nfrom multi_benchmark import preprocessing\nfrom tail_objective import temporal_groups\nfrom cluster_uncertainty import cluster_schemes\n\nSPECS=dict(ORIGINAL)\nSPECS[\'elliptic\']=dict(ref=\'ellipticco/elliptic-data-set\',files=[\'elliptic_txs_features.csv\',\'elliptic_txs_classes.csv\'],\n    numeric=[f\'f_{i:03d}\' for i in range(2,95)],categorical=[],label=\'is_illicit\',\n    kind=\'real anonymized Bitcoin transactions, known-label subset only\',unit=\'known-label Bitcoin transaction\',\n    licence=\'CC-BY-NC-ND-4.0\',source=\'https://arxiv.org/abs/1908.02591\',\n    excluded_reason=\'Transaction IDs, time and the final 72 backward/forward-neighbour features are excluded. Unknown labels are excluded, not treated as normal. Individual local-feature availability cannot be audited from anonymized columns.\')\n\ndef predictors(key):return feature_names(key) if key!=\'elliptic\' else SPECS[key][\'numeric\']\n\ndef processor(key):\n    if key!=\'elliptic\':return preprocessing(key)\n    return make_pipeline(SimpleImputer(strategy=\'median\',add_indicator=True,keep_empty_features=True),VarianceThreshold())\n\ndef sources(key,root):\n    if key!=\'elliptic\':return find_sources(key,root)\n    result=[]\n    for name in SPECS[key][\'files\']:\n        matches=list(Path(root).rglob(name))\n        if len(matches)!=1:raise ValueError(f\'Expected exactly one {name}\')\n        result.append(matches[0])\n    return result\n\ndef load_elliptic(paths,train_cap=150000,eval_cap=100000,chunksize=50000):\n    if train_cap<1 or eval_cap<1:raise ValueError(\'Positive cohort caps required\')\n    feature_path,label_path=map(Path,paths)\n    width=pd.read_csv(feature_path,header=None,nrows=1).shape[1]\n    if width!=167:raise ValueError(\'Expected ID plus the published 166 Elliptic features\')\n    labels=pd.read_csv(label_path,dtype={\'txId\':str,\'class\':str})\n    if list(labels.columns)!=[\'txId\',\'class\'] or labels.txId.isna().any() or labels.txId.duplicated().any():\n        raise ValueError(\'Expected unique present transaction IDs in labels\')\n    if not set(labels[\'class\'].unique()).issubset({\'1\',\'2\',\'unknown\'}):raise ValueError(\'Unexpected Elliptic label state\')\n    chunks=[]\n    for frame in pd.read_csv(feature_path,header=None,usecols=list(range(95)),dtype={0:str},chunksize=chunksize):\n        frame.columns=[\'txId\',\'__time\',*SPECS[\'elliptic\'][\'numeric\']]\n        chunks.append(frame)\n    full=pd.concat(chunks,ignore_index=True)\n    if full.txId.isna().any() or full.txId.duplicated().any():raise ValueError(\'Expected unique feature transaction IDs\')\n    if set(full.txId)!=set(labels.txId):raise ValueError(\'Feature and label transaction IDs differ\')\n    full=full.merge(labels,on=\'txId\',validate=\'one_to_one\',sort=False)\n    full[\'__row\']=np.arange(len(full),dtype=np.int64)\n    times=pd.to_numeric(full[\'__time\'],errors=\'raise\').to_numpy(dtype=float)\n    if not np.isfinite(times).all() or not np.equal(times,np.floor(times)).all():raise ValueError(\'Expected finite integer time steps\')\n    full[\'__time\']=times.astype(np.int64)\n    numeric=full[SPECS[\'elliptic\'][\'numeric\']].to_numpy(dtype=float)\n    if np.isinf(numeric).any():raise ValueError(\'Infinite local features are unsupported\')\n    unique=np.unique(times)\n    if len(unique)<8:raise ValueError(\'Insufficient time groups\')\n    edges=np.array([unique[int(len(unique)*q)] for q in [.55,.70,.85]])\n    label_counts={key:int(value) for key,value in full[\'class\'].value_counts().items()}\n    known=full[full[\'class\']!=\'unknown\'].copy();known[\'is_illicit\']=(known[\'class\']==\'1\').astype(int)\n    known[\'__priority\']=priorities(known[\'__row\'],20261112)\n    ids=np.searchsorted(edges,known[\'__time\'],side=\'right\')\n    frames=[];partitions={}\n    for i,name in enumerate(PARTS):\n        source=known[ids==i];cap=train_cap if i==0 else eval_cap\n        selected=source.nsmallest(cap,[\'__priority\',\'__row\']).sort_values(\'__row\').reset_index(drop=True)\n        if set(selected.is_illicit.unique())!={0,1}:raise ValueError(f\'Both known classes required in {name}\')\n        frames.append(selected)\n        partitions[name]=dict(source_rows=len(source),source_positive=int(source.is_illicit.sum()),\n            sampled_rows=len(selected),sampled_positive=int(selected.is_illicit.sum()),sampled_negative=int((selected.is_illicit==0).sum()),\n            time_min=int(selected.__time.min()),time_max=int(selected.__time.max()),\n            cohort_sha256=hashlib.sha256(selected[\'__row\'].to_numpy(dtype=np.int64).tobytes()).hexdigest())\n    for a,b in zip(frames,frames[1:]):\n        if a.__time.max()>=b.__time.min():raise ValueError(\'Temporal partitions overlap\')\n    spec=SPECS[\'elliptic\']\n    report=dict(dataset=\'elliptic\',dataset_ref=spec[\'ref\'],dataset_kind=spec[\'kind\'],unit=spec[\'unit\'],\n        source_url=spec[\'source\'],licence=spec[\'licence\'],observed_utc=datetime.now(timezone.utc).isoformat(),\n        files=[dict(name=p.name,sha256=file_sha256(p)) for p in [feature_path,label_path]],source_rows=len(full),\n        source_label_counts=label_counts,known_label_rows=len(known),unknown_label_rows=label_counts.get(\'unknown\',0),\n        predictor_allowlist=predictors(\'elliptic\'),raw_feature_columns_read=list(range(95)),\n        time_edges=edges.tolist(),time_unit=\'published time step\',partitions=partitions,\n        sampling=dict(seed=20261112,train_cap=train_cap,eval_cap=eval_cap,method=\'row-only stable hash priorities\'),\n        excluded_reason=spec[\'excluded_reason\'],source_missing_predictor_values={c:int(full[c].isna().sum()) for c in predictors(\'elliptic\')},\n        limitations=\'Real AML transfer task, not independent payment-fraud confirmation. Evaluation is conditional on known labels. Local anonymous feature semantics and provider normalization cannot establish point-in-time availability. Graph edges and forward-neighbour features are not used. No raw row export.\',\n        hypotheses=[\'IDs join one-to-one without missing or duplicate keys\',\'All four known-label partitions contain both classes\',\'Source timestamp groups do not cross partitions\',\'Only the published local feature positions excluding time enter predictors\'])\n    return frames,report\n\ndef load_data(key,paths,train_cap=150000,eval_cap=100000,chunksize=50000):\n    frames,report=load_elliptic(paths,train_cap,eval_cap,chunksize) if key==\'elliptic\' else load_cohort(key,paths,train_cap,eval_cap,chunksize=chunksize)\n    groups,edges=temporal_groups(frames[0][\'__time\']);y=frames[0][SPECS[key][\'label\']].to_numpy()\n    report[\'training_groups\']=[dict(group=int(g),rows=int((groups==g).sum()),positive=int(((groups==g)&(y==1)).sum()),negative=int(((groups==g)&(y==0)).sum())) for g in np.unique(groups)]\n    if any(not g[\'positive\'] or not g[\'negative\'] for g in report[\'training_groups\']):raise ValueError(\'A training group lacks one class\')\n    report[\'training_time_edges\']=edges.tolist()\n    report[\'uncertainty_cluster_counts\']={name:len(np.unique(values)) for name,values in cluster_schemes(key,frames[3]).items()}\n    report[\'holdout_status\']=\'First measured real Bitcoin AML transfer evaluation; settings frozen before scoring\' if key==\'elliptic\' else \'Consumed synthetic development evidence\'\n    return frames,report\n\nclass Inputs:\n    def __init__(self,key,train):\n        self.key=key;self.names=predictors(key);self.cats=SPECS[key][\'categorical\']\n        self.processor=processor(key).fit(train[self.names])\n        self.categories={c:sorted(train[c].dropna().astype(str).unique().tolist()) for c in self.cats}\n    def transform(self,frame,family):\n        if family not in {\'V6\',\'XGBoost\',\'CatBoost_native\',\'LightGBM_native\'}:raise ValueError(\'Unknown model family\')\n        data=frame[self.names].copy()\n        if family in {\'V6\',\'XGBoost\'}:\n            return np.ascontiguousarray(self.processor.transform(data),dtype=float)\n        for column in self.cats:\n            values=data[column].where(data[column].notna(),\'missing\').astype(str)\n            if family==\'CatBoost_native\':data[column]=values\n            elif family==\'LightGBM_native\':\n                values=values.where(values.isin(self.categories[column]),np.nan)\n                data[column]=pd.Categorical(values,categories=self.categories[column])\n            else:raise ValueError(\'Unknown model family\')\n        return data\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'dataset\',choices=SPECS);p.add_argument(\'root\');p.add_argument(\'output\');args=p.parse_args()\n    _,report=load_data(args.dataset,sources(args.dataset,args.root));Path(args.output).write_text(json.dumps(report,indent=2))\n', 'cluster_uncertainty.py': '"""Paired cluster bootstrap conditional on fitted models and fixed cutoffs."""\nimport numpy as np\n\ndef cluster_schemes(key,frame):\n    time=frame[\'__time\'].to_numpy(dtype=np.int64)\n    if key==\'sparkov\':\n        return {\'calendar_week\':time//(7*24*3600*10**9),\'card\':frame[\'cc_num\'].to_numpy(dtype=str)}\n    if key==\'paysim\':return {\'four_step_time_block\':time//4}\n    return {\'published_time_group\':time}\n\ndef paired_cluster_interval(y,candidate_flags,reference_flags,clusters,replicates=2000,seed=20261112,min_clusters=20):\n    y=np.asarray(y);a=np.asarray(candidate_flags,dtype=bool);b=np.asarray(reference_flags,dtype=bool);groups=np.asarray(clusters)\n    if y.ndim!=1 or a.shape!=y.shape or b.shape!=y.shape or groups.shape!=y.shape or set(np.unique(y))!={0,1}:\n        raise ValueError(\'Matching vectors and both binary classes required\')\n    if replicates<1 or min_clusters<2:raise ValueError(\'Invalid bootstrap settings\')\n    _,index=np.unique(groups,return_inverse=True);count=int(index.max()+1)\n    result=dict(cluster_count=count,replicates_requested=replicates,min_clusters=min_clusters,\n        observed_delta_recall=float(a[y==1].mean()-b[y==1].mean()),\n        observed_delta_fpr=float(a[y==0].mean()-b[y==0].mean()),\n        scope=\'Conditional on selected models and calibration cutoffs. Resamples clusters, not training or calibration data. Percentile intervals; no multiplicity correction or general dependence guarantee.\')\n    if count<min_clusters:return dict(result,status=\'unsupported\',reason=\'Too few observed clusters for the prespecified screen\')\n    pos=np.bincount(index,weights=(y==1),minlength=count);neg=np.bincount(index,weights=(y==0),minlength=count)\n    dp=np.bincount(index,weights=(a.astype(int)-b.astype(int))*(y==1),minlength=count)\n    dn=np.bincount(index,weights=(a.astype(int)-b.astype(int))*(y==0),minlength=count)\n    rng=np.random.default_rng(seed);recall=[];fpr=[]\n    for _ in range(replicates):\n        draw=rng.integers(0,count,size=count);p=pos[draw].sum();n=neg[draw].sum()\n        if p and n:recall.append(dp[draw].sum()/p);fpr.append(dn[draw].sum()/n)\n    if not recall:return dict(result,status=\'unsupported\',reason=\'No replicate contained both classes\')\n    return dict(result,status=\'evaluated\',valid_replicates=len(recall),\n        delta_recall_interval_95=np.quantile(recall,[.025,.975]).tolist(),\n        delta_fpr_interval_95=np.quantile(fpr,[.025,.975]).tolist())\n', 'tuned_models.py': '"""Four prespecified configurations per baseline; validation-only prefix selection."""\nimport time\nimport numpy as np\nfrom altcboost_v6 import ALTCBoostV6\nfrom detection_objective import selection_key\n\nGRIDS={\n \'CatBoost_native\':[dict(depth=4,positive_weight=1.),dict(depth=6,positive_weight=5.),dict(depth=6,positive_weight=\'train_ratio\'),dict(depth=8,positive_weight=10.)],\n \'XGBoost\':[dict(depth=4,positive_weight=1.),dict(depth=7,positive_weight=10.),dict(depth=5,positive_weight=\'train_ratio\'),dict(depth=9,positive_weight=5.)],\n \'LightGBM_native\':[dict(leaves=15,positive_weight=1.),dict(leaves=31,positive_weight=10.),dict(leaves=31,positive_weight=\'train_ratio\'),dict(leaves=63,positive_weight=5.)]}\n\ndef positive_weight(config,y):\n    y=np.asarray(y)\n    return float(np.sum(y==0)/np.sum(y==1)) if config[\'positive_weight\']==\'train_ratio\' else float(config[\'positive_weight\'])\n\ndef build_model(family,config,y,categories,seed=17,trees=100):\n    if family==\'V6\':return ALTCBoostV6(n_estimators=trees,random_state=seed)\n    weight=positive_weight(config,y)\n    if family==\'CatBoost_native\':\n        from catboost import CatBoostClassifier\n        return CatBoostClassifier(iterations=trees,depth=config[\'depth\'],scale_pos_weight=weight,\n            learning_rate=.06,random_seed=seed,cat_features=categories,thread_count=2,\n            bootstrap_type=\'Bayesian\',verbose=False,allow_writing_files=False)\n    if family==\'XGBoost\':\n        from xgboost import XGBClassifier\n        return XGBClassifier(n_estimators=trees,max_depth=config[\'depth\'],scale_pos_weight=weight,\n            learning_rate=.06,random_state=seed,n_jobs=2,tree_method=\'hist\',subsample=.9,colsample_bytree=.9)\n    if family==\'LightGBM_native\':\n        from lightgbm import LGBMClassifier\n        return LGBMClassifier(n_estimators=trees,num_leaves=config[\'leaves\'],scale_pos_weight=weight,\n            learning_rate=.06,random_state=seed,n_jobs=2,verbosity=-1,subsample=.9,subsample_freq=1,\n            colsample_bytree=.9,deterministic=True,force_col_wise=True)\n    raise ValueError(\'Unknown model family\')\n\ndef fit_model(family,model,x,y,xv,yv,groups):\n    if family==\'V6\':model.fit(x,y,xv,yv,groups)\n    else:model.fit(x,y)\n    return model\n\ndef raw_score(family,model,data,prefix):\n    if family not in {\'V6\',*GRIDS}:raise ValueError(\'Unknown model family\')\n    if family==\'V6\':\n        if prefix!=model.best_iteration_:raise ValueError(\'V6 prefix must match its frozen validation selection\')\n        return np.asarray(model.decision_function(data),dtype=float)\n    if prefix==0:return np.zeros(len(data),dtype=float)\n    if family==\'CatBoost_native\':return np.asarray(model.predict(data,prediction_type=\'RawFormulaVal\',ntree_end=prefix,thread_count=2),dtype=float)\n    if family==\'XGBoost\':\n        from xgboost import DMatrix\n        return np.asarray(model.get_booster().predict(DMatrix(data,nthread=2),output_margin=True,iteration_range=(0,prefix)),dtype=float)\n    if family==\'LightGBM_native\':return np.asarray(model.booster_.predict(data,raw_score=True,num_iteration=prefix,num_threads=2),dtype=float)\n    raise ValueError(\'Unknown model family\')\n\ndef staged_scores(family,model,data):\n    if family==\'CatBoost_native\':\n        # Use reference prefix calls so staged summation cannot change tied-score ranks.\n        for i in range(1,model.tree_count_+1):yield i,raw_score(family,model,data,i)\n    elif family==\'XGBoost\':\n        from xgboost import DMatrix\n        matrix=DMatrix(data,nthread=2);booster=model.get_booster()\n        for i in range(1,booster.num_boosted_rounds()+1):\n            yield i,np.asarray(booster.predict(matrix,output_margin=True,iteration_range=(0,i)),dtype=float)\n    elif family==\'LightGBM_native\':\n        # Direct prefix prediction avoids introducing a separate summation implementation.\n        for i in range(1,model.booster_.current_iteration()+1):yield i,raw_score(family,model,data,i)\n    else:raise ValueError(\'Unsupported staged model family\')\n\ndef select_prefix(family,model,data,y):\n    if family==\'V6\':return int(model.best_iteration_),selection_key(y,raw_score(family,model,data,model.best_iteration_))\n    best=selection_key(y,np.zeros(len(y)));prefix=0\n    for count,score in staged_scores(family,model,data):\n        key=selection_key(y,score)\n        if key>best:best=key;prefix=count\n    checked=selection_key(y,raw_score(family,model,data,prefix))\n    if not np.allclose(checked,best,rtol=0,atol=1e-12):\n        raise ValueError(\'Staged validation selection differs from direct prefix prediction\')\n    return prefix,checked\n\ndef tune_baselines(inputs,train,validation,label,groups,seed=17,trees=100):\n    """Only training and validation frames enter this selection function."""\n    y=train[label].to_numpy(dtype=int);yv=validation[label].to_numpy(dtype=int)\n    chosen={};trials=[]\n    for family,grid in GRIDS.items():\n        x=inputs.transform(train,family);xv=inputs.transform(validation,family);best=None\n        for trial,config in enumerate(grid):\n            model=build_model(family,config,y,inputs.cats,seed,trees)\n            start=time.perf_counter();fit_model(family,model,x,y,xv,yv,groups);fit_s=time.perf_counter()-start\n            start=time.perf_counter();prefix,key=select_prefix(family,model,xv,yv);selection_s=time.perf_counter()-start\n            record=dict(family=family,trial=trial,config=config,resolved_positive_weight=positive_weight(config,y),\n                prefix=prefix,validation_key=list(key),fit_s=fit_s,selection_s=selection_s)\n            trials.append(record)\n            if best is None or key>best[\'key\']:best=dict(model=model,config=config,prefix=prefix,key=key,record=record)\n        chosen[family]=best\n        print(\'Validation selected\',family,best[\'config\'],\'prefix\',best[\'prefix\'],flush=True)\n    return chosen,trials\n', 'v7_research.py': '"""Mechanism experiments on training/validation only; no later-interval scoring."""\nimport argparse\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom threadpoolctl import threadpool_limits\nfrom altcboost_v6 import ALTCBoostV6\nfrom altcboost_v7 import ALTCBoostV7\nfrom validation_data import Inputs,SPECS,load_data,sources\nfrom tuned_models import build_model,fit_model,raw_score\nfrom tail_objective import temporal_groups\nfrom detection_objective import selection_key,empirical_cutoff\nfrom evaluation import metrics\n\nMODES=[\'zero\',\'clipped\',\'shrunk\',\'adaptive_diagonal\',\'adaptive_joint\']\n\ndef fit_modes(key,train,validation,references=None,trees=100):\n    inputs=Inputs(key,train);label=SPECS[key][\'label\']\n    y=train[label].to_numpy(dtype=int);yv=validation[label].to_numpy(dtype=int)\n    x=inputs.transform(train,\'V6\');xv=inputs.transform(validation,\'V6\');groups,_=temporal_groups(train[\'__time\'])\n    candidates=[(\'V6_fixed\',ALTCBoostV6(n_estimators=trees))]\n    candidates.extend((\'V7_\'+mode,ALTCBoostV7(prior_mode=mode,n_estimators=trees)) for mode in MODES)\n    rows=[];models=[]\n    def record(name,score,metadata):\n        key_value=selection_key(yv,score)\n        for alpha in [.0005,.001,.01]:\n            cutoff=empirical_cutoff(yv,score,alpha)\n            rows.append(dict(dataset=key,method=name,alpha=alpha,\n                validation_mean_recall=key_value[0],validation_mean_fpr=-key_value[1],\n                threshold=cutoff,**metrics(yv,score,cutoff,alpha),**metadata))\n        print(key,name,\'validation key\',key_value,flush=True)\n    for name,model in candidates:\n        with threadpool_limits(limits=2):\n            start=time.perf_counter();model.fit(x,y,xv,yv,groups);duration=time.perf_counter()-start\n        score=model.decision_function(xv)\n        weight=model.prior_weights_[model.best_iteration_] if hasattr(model,\'prior_weights_\') else 1.\n        metadata=dict(fit_s=duration,selected_prefix=int(model.best_iteration_),selected_prior_weight=float(weight))\n        record(name,score,metadata)\n        models.append(dict(method=name,**metadata,training_loss=model.training_loss_,\n            prior_weights=getattr(model,\'prior_weights_\',None),step_diagnostics=model.step_diagnostics_))\n    for family,reference in (references or {}).items():\n        model=build_model(family,reference[\'config\'],y,inputs.cats,17,trees)\n        with threadpool_limits(limits=2):\n            start=time.perf_counter();fit_model(family,model,inputs.transform(train,family),y,inputs.transform(validation,family),yv,groups)\n            duration=time.perf_counter()-start\n        prefix=int(reference[\'prefix\']);score=raw_score(family,model,inputs.transform(validation,family),prefix)\n        checked=selection_key(yv,score)\n        if not np.allclose(checked,reference[\'validation_key\'],rtol=0,atol=1e-12):raise ValueError(\'Frozen reference validation scores did not reproduce\')\n        record(family,score,dict(fit_s=duration,selected_prefix=prefix,selected_prior_weight=None))\n    return rows,models\n\ndef run(key,paths,output,references=None,trees=100,train_cap=150000,eval_cap=100000,expected_cohorts=None,expected_files=None):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    frames,discovery=load_data(key,paths,train_cap,eval_cap)\n    if expected_cohorts:\n        for name,expected in expected_cohorts.items():\n            if discovery[\'partitions\'][name][\'cohort_sha256\']!=expected:raise ValueError(\'Research cohort differs from frozen reference\')\n    if expected_files and {item[\'name\']:item[\'sha256\'] for item in discovery[\'files\']}!=expected_files:\n        raise ValueError(\'Source file hash differs from frozen reference\')\n    discovery[\'holdout_status\']=\'Consumed source; this experiment fits and scores training/validation only\'\n    rows,models=fit_modes(key,frames[0],frames[1],references,trees)\n    pd.DataFrame(rows).to_csv(output/\'validation.csv\',index=False)\n    (output/\'models.json\').write_text(json.dumps(models,indent=2))\n    (output/\'discovery.json\').write_text(json.dumps(discovery,indent=2))\n    manifest=dict(dataset=key,scope=\'Adaptive development experiment. Fit and score training/validation only; no calibration or final prediction is computed.\',\n        source_url=SPECS[key][\'source\'],unit=SPECS[key][\'unit\'],dataset_kind=SPECS[key][\'kind\'],\n        modes=[\'V6_fixed\',*[\'V7_\'+mode for mode in MODES]],tree_cap=trees,seed=17,\n        prior_controls=dict(clipped_mapping=\'4*tanh(raw_prior/4)\',shrunk_coefficient=.25),\n        candidate=\'Box-constrained prior/tree block updates using exact directional curvature of the temporal negative-tail objective\',\n        limitations=\'Validation chooses prefixes and is also used to compare these hypotheses; scores are optimistic development evidence. Existing datasets informed the redesign. No novelty, general superiority, independent confirmation, future FPR control or global optimizer claim. Zero-start can encounter nonsmooth tail ties. Fully corrective boosting and variable-projection methods are related prior work.\')\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    return rows,models\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'dataset\',choices=SPECS);p.add_argument(\'root\');p.add_argument(\'output\');p.add_argument(\'--references\')\n    args=p.parse_args();references=json.loads(Path(args.references).read_text()) if args.references else None\n    run(args.dataset,sources(args.dataset,args.root),args.output,references)\n', 'v7_report.py': '"""Report a mechanism experiment without relabelling validation as confirmation."""\nimport json\nfrom pathlib import Path\nimport pandas as pd\n\ndef write_v7_report(source,destination):\n    source=Path(source);destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)\n    rows=pd.read_csv(source/\'validation.csv\');manifest=json.loads((source/\'manifest.json\').read_text())\n    lines=[f"# Prior-revision experiment: {manifest[\'dataset\']}",\'\',\n        \'This is a training/validation mechanism experiment. The same validation labels select model prefixes and compare the hypotheses. These scores are development evidence and do not establish independent confirmation or novelty.\',\'\',\n        \'| Method | Mean recall at two strict validation budgets | Caught at 0.1% | False positives at 0.1% | Selected prior weight | Selected prefix |\',\n        \'| --- | ---: | ---: | ---: | ---: | ---: |\']\n    for row in rows[(rows.alpha-.001).abs()<1e-12].to_dict(\'records\'):\n        weight=f"{row[\'selected_prior_weight\']:.4f}" if pd.notna(row[\'selected_prior_weight\']) else \'N/A\'\n        lines.append(f"| {row[\'method\']} | {row[\'validation_mean_recall\']:.2%} | {int(row[\'tp\'])} | {int(row[\'fp\'])} | {weight} | {int(row[\'selected_prefix\'])} |")\n    lines.extend([\'\',\'The primary mechanism question is whether the coupled update improves over zero, clipping, fixed shrinkage and diagonal-block controls. A simpler control matching or beating it weakens the claim for the extra mechanism. The 0.05% and 1% validation results remain in validation.csv.\',\'\',\n        \'## For whoever builds this\',\'\',manifest[\'scope\'],manifest[\'limitations\'],\'\',\n        \'The prior weight is constrained to [0,1]. Every accepted update is checked against the actual empirical training loss. Exact directional curvature concerns a two-coordinate restriction away from tail membership ties; the tree structure still uses a diagonal Newton approximation. Neither this check nor a validation gain proves future low-FPR performance.\',\n        \'\', \'Decision still needed: the research lead must assess whether the coupled update earns its complexity over the controls and the closest published methods. A revised method needs a new confirmation protocol before a publication claim.\'])\n    target=destination/\'review.md\';target.write_text(\'\\n\'.join(lines)+\'\\n\');return target\n', 'watch_kaggle.py': '"""Retrieve a completed Kaggle run and build its aggregate report."""\nimport argparse\nimport json\nimport re\nimport subprocess\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom hosted_summary import write_report\n\nCLI=\'/Users/apple/.local/bin/kaggle\'\n\ndef parse_status(text):\n    match=re.search(r\'KernelWorkerStatus\\.([A-Z_]+)\',text)\n    if not match:\n        raise ValueError(\'Kaggle returned no recognized worker status\')\n    state=match.group(1)\n    if state not in {\'QUEUED\',\'RUNNING\',\'COMPLETE\',\'ERROR\',\'CANCEL_REQUESTED\',\'CANCEL_ACKNOWLEDGED\',\'NEW_SCRIPT\'}:\n        raise ValueError(\'Unknown Kaggle worker status\')\n    return state\n\ndef run_cli(args):\n    return subprocess.run([CLI,*args],text=True,capture_output=True,check=True,timeout=180).stdout\n\ndef tick(root,kernel,runner=run_cli,kind=\'fraud\'):\n    if kind not in {\'fraud\',\'speed\',\'detection\',\'multi\',\'validation\',\'v7\'}:raise ValueError(\'Unknown experiment kind\')\n    root=Path(root); root.mkdir(parents=True,exist_ok=True)\n    state=parse_status(runner([\'kernels\',\'status\',kernel]))\n    record=dict(kernel=kernel,state=state,checked_utc=datetime.now(timezone.utc).isoformat())\n    if state==\'COMPLETE\':\n        out=root/\'kaggle-output\';out.mkdir(exist_ok=True)\n        runner([\'kernels\',\'output\',kernel,\'-p\',str(out)])\n        pattern={\'v7\':\'research_results/validation.csv\',\'validation\':\'validation_results/results.csv\',\'multi\':\'multi_results/results.csv\',\'speed\':\'speed_results/timing.csv\',\'detection\':\'quality_results/results.csv\',\n                 \'fraud\':\'altcboost_results/results.csv\'}[kind]\n        results=list(out.rglob(pattern))\n        if len(results)!=1:\n            raise ValueError(\'Expected one completed benchmark results file\')\n        if kind==\'v7\':\n            from v7_report import write_v7_report\n            write_v7_report(results[0].parent,root/\'hosted-report\')\n        elif kind==\'validation\':\n            from validation_report import write_validation_report\n            write_validation_report(results[0].parent,root/\'hosted-report\')\n        elif kind==\'multi\':\n            from multi_report import write_multi_report\n            write_multi_report(results[0].parent,root/\'hosted-report\')\n        elif kind==\'speed\':\n            from speed_report import write_speed_report\n            write_speed_report(results[0].parent,root/\'hosted-report\')\n        elif kind==\'detection\':\n            from detection_report import write_detection_report\n            write_detection_report(results[0].parent,root/\'hosted-report\')\n        else:\n            discovery=results[0].with_name(\'discovery.json\')\n            write_report(results[0],discovery,root/\'hosted-report\')\n        record[\'report\']=str(root/\'hosted-report/review.md\')\n    elif state in {\'ERROR\',\'CANCEL_ACKNOWLEDGED\'}:\n        (root/\'kaggle-failure.log\').write_text(runner([\'kernels\',\'logs\',kernel]))\n    (root/\'kaggle-status.json\').write_text(json.dumps(record,indent=2))\n    return record\n\ndef watch(root,kernel,interval=45,kind=\'fraud\'):\n    root=Path(root)\n    while True:\n        try:\n            record=tick(root,kernel,kind=kind)\n            print(json.dumps(record),flush=True)\n            if record[\'state\'] in {\'COMPLETE\',\'ERROR\',\'CANCEL_ACKNOWLEDGED\'}:\n                return record\n        except Exception as error:\n            # Retry retrieval failures; a failed HTTP request is not a failed training run.\n            record=dict(checked_utc=datetime.now(timezone.utc).isoformat(),\n                        retrieval_error=str(error))\n            (root/\'kaggle-retrieval-error.json\').write_text(json.dumps(record,indent=2))\n            print(json.dumps(record),flush=True)\n        time.sleep(interval)\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'kernel\');parser.add_argument(\'--root\',default=\'..\')\n    parser.add_argument(\'--kind\',choices=[\'fraud\',\'speed\',\'detection\',\'multi\',\'validation\',\'v7\'],default=\'fraud\')\n    args=parser.parse_args();watch(args.root,args.kernel,kind=args.kind)\n', 'hosted_summary.py': '"""Summarize completed hosted runs without pooling repeated holdouts."""\nimport argparse\nimport json\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nMETHODS={\'Logistic\',\'XGBoost\',\'LightGBM\',\'CatBoost\',\'ALTCBoostV2\',\n         \'ALTCBoostV3_original\',\'ALTCBoostV3_aligned\'}\n\ndef summarize(frame, alpha=.001, expected_seeds=(17,42)):\n    selected=frame[np.isclose(frame[\'alpha\'],alpha)].copy()\n    if set(selected[\'seed\'])!=set(expected_seeds):\n        raise ValueError(\'Hosted results do not contain all planned seeds\')\n    if selected.duplicated([\'method\',\'seed\']).any():\n        raise ValueError(\'Duplicate method and seed results\')\n    for seed in expected_seeds:\n        if set(selected[selected.seed==seed].method)!=METHODS:\n            raise ValueError(\'Hosted results do not contain all planned model families\')\n    if set(selected.status)!={\'evaluated\'}:\n        raise ValueError(\'Some operating points were not evaluated\')\n    selected[\'test_positive_count\']=selected.tp+selected.fn\n    selected[\'test_negative_count\']=selected.fp+selected.tn\n    if selected.test_positive_count.nunique()!=1 or selected.test_negative_count.nunique()!=1:\n        raise ValueError(\'Holdout sizes differ across repeated runs\')\n    if not np.allclose(selected.recall,selected.tp/selected.test_positive_count):\n        raise ValueError(\'Recall disagrees with saved counts\')\n    if not np.allclose(selected.fpr,selected.fp/selected.test_negative_count):\n        raise ValueError(\'False-positive rate disagrees with saved counts\')\n    summary=[]\n    for method, group in selected.groupby(\'method\',sort=False):\n        summary.append(dict(method=method,recall_min=float(group.recall.min()),\n            recall_max=float(group.recall.max()),tp_min=int(group.tp.min()),tp_max=int(group.tp.max()),\n            fp_min=int(group.fp.min()),fp_max=int(group.fp.max()),\n            fpr_min=float(group.fpr.min()),fpr_max=float(group.fpr.max()),\n            meets_empirical_budget_each_seed=bool((group.fpr<=alpha).all()),\n            scoring_p50_us_median=float(group.single_p50_us.median()),\n            fit_seconds_median=float(group.fit_s.median())))\n    return summary,selected\n\ndef write_report(results,discovery,output):\n    results,discovery,output=map(Path,[results,discovery,output])\n    frame=pd.read_csv(results); provenance=json.loads(discovery.read_text())\n    summary,selected=summarize(frame)\n    test=provenance[\'splits\'][\'test\']\n    output.mkdir(parents=True,exist_ok=True)\n    selected.to_csv(output/\'operating-point-0.001.csv\',index=False)\n    evidence=dict(source=str(results.resolve()),discovery_source=str(discovery.resolve()),\n        observed_utc=datetime.now(timezone.utc).isoformat(),\n        scope=\'Kaggle-hosted CPU run; fixed temporal holdout; two training seeds; calibration target FPR 0.001\',\n        unit=\'transaction row; repeated seeds use the same holdout\',models=summary)\n    (output/\'summary.json\').write_text(json.dumps(evidence,indent=2))\n    table=[\'| Model | Fraud caught per seed | False alarms per seed | Recall range |\',\n           \'|---|---:|---:|---:|\']\n    for row in summary:\n        table.append(f"| {row[\'method\']} | {row[\'tp_min\']}-{row[\'tp_max\']} | {row[\'fp_min\']}-{row[\'fp_max\']} | {row[\'recall_min\']:.1%}-{row[\'recall_max\']:.1%} |")\n    report=f\'\'\'# Kaggle-hosted benchmark results\n\nUse these results to decide what to test on a fresh time period. They do not establish a production choice. The comparison used a private CPU notebook, two training seeds and a 70-tree limit.\n\nThe calibration target was a 0.1% false-positive rate. The later test interval had {test[\'positive\']} fraud cases and {test[\'negative\']:,} normal transaction rows. Every model used the same calibration rule. The table shows the range across seeds on that one holdout. Counts are not added across seeds.\n\n{chr(10).join(table)}\n\nChanges in versions, tree budget and hardware mean that latency and quality can differ from the earlier local pilot. Model scoring excludes feature retrieval and network overhead. The data provide no card identities or label-arrival times, so entity independence and realistic label delays remain unverified. A calibration target does not guarantee the same false-positive rate on future traffic.\n\n## For whoever builds this\n\nSkip this section unless you are running or changing the code.\n\nThe source is `{results.resolve()}`. Dataset provenance is in `{discovery.resolve()}`. Summary calculations are saved in `summary.json` and `operating-point-0.001.csv` beside this report. Regenerate with `python hosted_summary.py RESULTS_CSV DISCOVERY_JSON OUTPUT_DIRECTORY`.\n\nThe notebook runs `tests/test_unit.py` and `tests/test_integration.py` before training. Hosted logs contain the actual pass counts. The separately tested CLI packaging code runs locally and is not part of hosted model execution.\n\nDecisions still needed: Jitendra must choose alert capacity and the operational false-positive budget. The fraud-data owner must provide a later holdout with identities and label-arrival information before a deployment assessment. Any redesign based on this test interval needs another final holdout.\n\'\'\'\n    (output/\'review.md\').write_text(report)\n    return evidence\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'results\');p.add_argument(\'discovery\');p.add_argument(\'output\')\n    args=p.parse_args();print(json.dumps(write_report(args.results,args.discovery,args.output),indent=2))\n', 'tests/test_multi_unit.py': "import numpy as np\nimport pandas as pd\nimport pytest\nfrom multi_data import SPECS,feature_names,load_cohort,priorities,read_chunks\nfrom multi_benchmark import preprocessing\n\ndef fixture_csv(tmp_path,n=4000):\n    row=np.arange(n)\n    frame=pd.DataFrame(dict(step=row//40,type=np.where(row%2,'PAYMENT','TRANSFER'),\n        amount=(row%37).astype(float),oldbalanceOrg=(row%53).astype(float),\n        oldbalanceDest=(row%17).astype(float),nameOrig=['synthetic'+str(i%50) for i in row],\n        isFraud=(row%13==0).astype(int)))\n    path=tmp_path/'synthetic.csv';frame.to_csv(path,index=False);return path,frame\n\ndef test_sampling_is_chunk_invariant_and_label_blind(tmp_path):\n    path,frame=fixture_csv(tmp_path)\n    first,a=load_cohort('paysim',[path],300,250,chunksize=137)\n    _,b=load_cohort('paysim',[path],300,250,chunksize=1001)\n    assert a['partitions']==b['partitions']\n    frame.isFraud=1-frame.isFraud;frame.to_csv(path,index=False)\n    second,_=load_cohort('paysim',[path],300,250,chunksize=501)\n    assert all(np.array_equal(x['__row'],y['__row']) for x,y in zip(first,second))\n    assert all(x['__time'].max()<y['__time'].min() for x,y in zip(first,first[1:]))\n\ndef test_labels_checked_before_cast(tmp_path):\n    path,frame=fixture_csv(tmp_path);frame.loc[0,'isFraud']=2;frame.to_csv(path,index=False)\n    with pytest.raises(ValueError,match='binary'):list(read_chunks('paysim',[path]))\n\n@pytest.mark.parametrize('key',list(SPECS))\ndef test_predictor_allowlist_excludes_targets_identifiers_and_future_fields(key):\n    assert not set(feature_names(key))&{'isFraud','is_fraud','fraud_bool','nameOrig','cc_num',\n        'newbalanceOrig','newbalanceDest','isFlaggedFraud','month','device_fraud_count'}\n\ndef test_training_preprocessing_does_not_learn_test_values(tmp_path):\n    _,frame=fixture_csv(tmp_path);names=feature_names('paysim');processor=preprocessing('paysim')\n    train=frame[names].copy();train.loc[0,'amount']=np.nan\n    x=processor.fit_transform(train)\n    test=train.iloc[:2].copy();test['type']='UNSEEN';test['amount']=np.nan\n    transformed=processor.transform(test)\n    assert transformed.shape[1]==x.shape[1] and np.isfinite(transformed).all()\n    assert processor.named_steps['prepare'].named_transformers_['numeric'].statistics_[0]==np.nanmedian(train.amount)\n\ndef test_priorities_are_row_only():\n    assert np.array_equal(priorities(np.arange(10),17),np.concatenate([priorities(np.arange(4),17),priorities(np.arange(4,10),17)]))\n\ndef test_sparkov_official_time_overlap_is_rejected(tmp_path):\n    frame=pd.DataFrame(dict(trans_date_trans_time=pd.date_range('2020-01-01',periods=100,freq='h').astype(str),\n        amt=np.arange(100),city_pop=100,category='synthetic',cc_num='synthetic',is_fraud=np.arange(100)%2))\n    first=tmp_path/'train.csv';second=tmp_path/'test.csv';frame.to_csv(first,index=False);frame.to_csv(second,index=False)\n    with pytest.raises(ValueError,match='overlap'):load_cohort('sparkov',[first,second])\n", 'tests/test_tail_integration.py': "import numpy as np\nimport pytest\nfrom sklearn.datasets import make_classification\nfrom altcboost_v5 import ALTCBoostV5\n\ndef example():\n    x,y=make_classification(n_samples=1600,n_features=6,n_redundant=0,weights=[.9,.1],random_state=72)\n    return x,y,np.arange(1000)%4\n\ndef test_fit_accepts_only_descending_steps_and_returns_selected_model():\n    x,y,groups=example()\n    model=ALTCBoostV5(n_estimators=8,min_samples_leaf=15).fit(x[:1000],y[:1000],x[1000:1300],y[1000:1300],groups)\n    assert len(model.training_loss_)>1 and np.all(np.diff(model.training_loss_)<0)\n    assert 0<=model.best_iteration_<=len(model.steps_)<=8\n    z=model.scaler_.transform(x[1300:]);expected=model.linear_.decision_function(z)\n    for tree,rate in model.steps_[:model.best_iteration_]:expected+=rate*tree.predict(z)\n    assert np.allclose(expected,model.decision_function(x[1300:]))\n    assert np.allclose(model.predict_proba(x[1300:]).sum(axis=1),1.)\n    with pytest.raises(ValueError,match='feature count'):model.predict_proba(np.ones((2,5)))\n\ndef test_validation_labels_cannot_change_first_training_update():\n    x,y,groups=example()\n    models=[ALTCBoostV5(n_estimators=1,min_samples_leaf=15).fit(x[:1000],y[:1000],x[1000:1300],v,groups)\n            for v in [y[1000:1300],1-y[1000:1300]]]\n    assert models[0].training_loss_==models[1].training_loss_\n    assert np.array_equal(models[0].steps_[0][0].tree_.value,models[1].steps_[0][0].tree_.value)\n\ndef test_refit_clears_previous_steps():\n    x,y,groups=example();model=ALTCBoostV5(n_estimators=3,min_samples_leaf=15)\n    model.fit(x[:1000],y[:1000],x[1000:1300],y[1000:1300],groups)\n    model.n_estimators=1;model.fit(x[:1000],y[:1000],x[1000:1300],y[1000:1300],groups)\n    assert len(model.steps_)==1 and len(model.training_loss_)==2\n", 'tests/test_prior_block_unit.py': "import numpy as np\nimport pytest\nfrom prior_block import directional_derivatives,block_proposals\nfrom tail_objective import pairwise_tail_objective\n\n@pytest.mark.parametrize('anchor',[0.,.2,1.])\ndef test_full_directional_curvature_matches_numeric_mixed_derivatives(anchor):\n    y=np.tile([0,0,0,1,1],2);groups=np.repeat([0,1],5)\n    score=np.array([-.7,.3,1.2,-.2,.9,.2,-.5,1.8,.4,-.8])\n    directions=np.random.default_rng(12).normal(size=(len(y),2));eps=1e-4\n    value,g,h=directional_derivatives(y,score,groups,directions,tail_fraction=.5,anchor_weight=anchor)\n    loss=lambda delta:pairwise_tail_objective(y,score+directions@delta,groups,tail_fraction=.5,anchor_weight=anchor)[0]\n    for i in range(2):\n        step=np.eye(2)[i]*eps\n        assert g[i]==pytest.approx((loss(step)-loss(-step))/(2*eps),abs=2e-8)\n        assert h[i,i]==pytest.approx((loss(step)-2*value+loss(-step))/eps**2,abs=2e-6)\n    first=np.array([eps,0]);second=np.array([0,eps])\n    mixed=(loss(first+second)-loss(first-second)-loss(-first+second)+loss(-first-second))/(4*eps**2)\n    assert h[0,1]==pytest.approx(mixed,abs=2e-6)\n    assert np.linalg.eigvalsh(h).min()>=-1e-10\n\ndef test_proposals_respect_prior_and_tree_boxes_even_at_boundaries():\n    for weight in [0.,.4,1.]:\n        proposals=block_proposals([10.,-3.],[[2.,.3],[.3,1.]],weight)\n        for _,step in proposals:\n            assert 0<=weight+step[0]<=1 and 0<=step[1]<=.08\n\ndef test_cross_curvature_changes_joint_proposal():\n    g=[.05,-.02];h=[[2.,1.2],[1.2,1.]]\n    full=block_proposals(g,h,.5)[-1][1]\n    diagonal=block_proposals(g,h,.5,diagonal=True)[-1][1]\n    assert not np.allclose(full,diagonal)\n\ndef test_bounded_prior_is_monotone_and_cannot_extrapolate_without_limit():\n    from prior_block import bounded_prior\n    score=np.linspace(-1000,1000,10001);mapped=bounded_prior(score)\n    assert np.all(np.diff(mapped)>=0) and np.max(np.abs(mapped))<=4\n    assert bounded_prior([0])[0]==0\n", 'tests/test_v7_integration.py': "import numpy as np\nimport pytest\nfrom altcboost_v6 import ALTCBoostV6\nfrom altcboost_v7 import ALTCBoostV7\nfrom detection_objective import selection_key\nfrom test_tail_integration import example\n\n@pytest.mark.parametrize('mode',['fixed','zero','clipped','shrunk','adaptive_diagonal','adaptive_joint'])\ndef test_prior_modes_keep_descent_and_predict_the_selected_weighted_prefix(mode):\n    x,y,groups=example();model=ALTCBoostV7(prior_mode=mode,n_estimators=7,min_samples_leaf=15)\n    model.fit(x[:1000],y[:1000],x[1000:1300],y[1000:1300],groups)\n    assert np.all(np.diff(model.training_loss_)<0)\n    assert all(0<=weight<=1 for weight in model.prior_weights_)\n    z=model.scaler_.transform(x[1300:]);expected=model.prior_weights_[model.best_iteration_]*model._prior(z)\n    for tree,rate in model.steps_[:model.best_iteration_]:expected+=rate*tree.predict(z)\n    assert np.allclose(expected,model.decision_function(x[1300:]),rtol=0,atol=1e-12)\n    assert selection_key(y[1000:1300],model.decision_function(x[1000:1300]))==model.validation_key_\n    assert np.allclose(model.predict_proba(x[1300:]).sum(axis=1),1.)\n\ndef test_fixed_prior_control_reproduces_v6():\n    x,y,groups=example()\n    models=[ALTCBoostV6(n_estimators=7,min_samples_leaf=15),ALTCBoostV7(prior_mode='fixed',n_estimators=7,min_samples_leaf=15)]\n    for model in models:model.fit(x[:1000],y[:1000],x[1000:1300],y[1000:1300],groups)\n    assert models[0].best_iteration_==models[1].best_iteration_\n    assert np.allclose(models[0].decision_function(x[1300:]),models[1].decision_function(x[1300:]),rtol=0,atol=1e-12)\n\ndef test_refit_resets_prior_coefficient_history():\n    x,y,groups=example();model=ALTCBoostV7(n_estimators=3,min_samples_leaf=15)\n    model.fit(x[:1000],y[:1000],x[1000:1300],y[1000:1300],groups)\n    model.n_estimators=1;model.fit(x[:1000],y[:1000],x[1000:1300],y[1000:1300],groups)\n    assert len(model.prior_weights_)==len(model.steps_)+1<=2\n", 'tests/test_v7_research_unit.py': "from v7_research import MODES\n\ndef test_mechanism_plan_includes_simpler_starting_model_controls():\n    assert set(MODES)=={'zero','clipped','shrunk','adaptive_diagonal','adaptive_joint'}\n", 'tests/test_v7_research_integration.py': "import json\nfrom pathlib import Path\nimport numpy as np\nfrom v7_research import run,fit_modes\nfrom validation_data import load_data\nfrom test_multi_unit import fixture_csv\nfrom watch_kaggle import tick\n\ndef test_research_runner_and_report_preserve_validation_scope(tmp_path):\n    path,_=fixture_csv(tmp_path,8000);source=tmp_path/'kaggle-output/research_results'\n    rows,models=run('paysim',[path],source,trees=3,train_cap=1000,eval_cap=2000)\n    assert len(rows)==18 and len(models)==6\n    assert all(np.all(np.diff(model['training_loss'])<0) for model in models)\n    record=tick(tmp_path,'synthetic/run/1',runner=lambda args:'KernelWorkerStatus.COMPLETE' if args[1]=='status' else '',kind='v7')\n    text=Path(record['report']).read_text();assert 'development evidence' in text\n    assert 'V7_adaptive_joint' in text and 'V7_clipped' in text\n    assert 'synthetic0' not in ''.join(path.read_text() for path in source.iterdir())\n\ndef test_mechanism_fit_never_receives_calibration_or_final_frames(tmp_path,monkeypatch):\n    import v7_research\n    path,_=fixture_csv(tmp_path,8000);frames,discovery=load_data('paysim',[path],1000,2000)\n    class Poison:\n        def __getitem__(self,key):raise AssertionError('Future frame used for fitting or scoring')\n    monkeypatch.setattr(v7_research,'load_data',lambda *args:([frames[0],frames[1],Poison(),Poison()],discovery))\n    run('paysim',[path],tmp_path/'output',trees=2)\n\ndef test_source_mismatch_stops_before_mechanism_fitting(tmp_path,monkeypatch):\n    import pytest\n    import v7_research\n    path,_=fixture_csv(tmp_path,8000)\n    monkeypatch.setattr(v7_research,'fit_modes',lambda *args:(_ for _ in ()).throw(AssertionError('Fitting should not begin')))\n    for guards,message in [({'expected_cohorts':{'train':'wrong source'}},'cohort differs'),\n                           ({'expected_files':{path.name:'wrong hash'}},'file hash differs')]:\n        with pytest.raises(ValueError,match=message):\n            run('paysim',[path],tmp_path/'output',trees=2,**guards)\n"}
for name,text in files.items():
 p=root/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(text)
os.chdir(root)
sys.path[:0]=[str(root),str(root/"tests")]


In [ ]:
import subprocess
subprocess.run([sys.executable,"-m","pytest","-q","tests/test_prior_block_unit.py","tests/test_v7_research_unit.py"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","tests/test_v7_integration.py","tests/test_v7_research_integration.py"],check=True)


In [ ]:
output=Path("/kaggle/working/research_results");output.mkdir(exist_ok=True)
(output/"prespecified-plan.json").write_text(json.dumps({'created_utc': '2026-11-13T02:35:35.250048+00:00', 'hypothesis': 'A revisable prior plus a new-tree coordinate can improve strict-tail validation performance over fixed, zero and statically modified priors', 'mechanisms': ['V6_fixed', 'V7_zero', 'V7_clipped', 'V7_shrunk', 'V7_adaptive_diagonal', 'V7_adaptive_joint'], 'fixed_settings': {'trees': 100, 'max_leaves': 17, 'min_samples_leaf': 40, 'learning_rate': 0.08, 'leaf_step_bound': 5.0, 'tail_fraction': 0.01, 'anchor_weight': 0.2, 'group_temperature': 0.25, 'block_damping': 0.001, 'seed': 17}, 'controls': {'clipped': '4*tanh(raw_prior/4)', 'shrunk': 'fixed prior coefficient 0.25', 'zero': 'zero starting score; same nonsmooth target loss'}, 'adaptive_update': 'Box-constrained prior coefficient in [0,1] and new-tree coefficient in [0,0.08]; normalized damped exact two-coordinate Newton proposal; compare with tree-only and prior-only proposals using actual-loss backtracking', 'evaluation': 'Training and original validation only. No calibration or final frame is fitted, transformed for prediction or scored after source discovery. Validation selects prefixes and compares hypotheses, so outcomes are optimistic adaptive development.', 'success_questions': ['Does adaptive_joint improve beyond the best simple starting-score control?', 'Does cross-coordinate curvature help beyond adaptive_diagonal?', 'Does the learned prior weight vary usefully by data regime instead of merely duplicating zero start?'], 'falsifiers': ['If simpler controls match or beat the joint update, do not attribute gains to the extra mechanism.', 'If diagonal updates match the full update, do not claim cross-curvature is responsible.', 'Do not call clipping, coefficient correction, partial-AUC or group DRO individually novel.'], 'reference_policy': 'Refit previously selected native baseline configurations at frozen prefixes; require their validation keys to reproduce', 'reference_cohorts': {'elliptic': {'training_cohort': '9a2c63663f0c4fc11a04d34d7324b88306e2439f8ab5d29c7cf12e65ba46c1b0', 'validation_cohort': '72e3934b48976b5b1da6ffacf611cb295ab17e6f8614f402b96536e74787fcb4', 'expected_v6_validation_key': [0.00048590864917395527, -0.0007232401157184186, 0.9311178155927057], 'source_files': {'elliptic_txs_features.csv': 'fd7f83573443c9e302e371d3f110e3b6224160f5d1ed8a287757936127800ff0', 'elliptic_txs_classes.csv': '93e2e7b2405c735ba752bf6ba06b947561deddd1f5a8fc91e46f6a4c0e439493'}}, 'paysim': {'training_cohort': '72ee3c097751a7850a42f82d2b5b443b4d8c75a04a0a65dbc6aaabde212de093', 'validation_cohort': '33bea9fe34b374131b57a0b8aec62e4956d9e3f3e275cf54438ddfd109aa2e80', 'expected_v6_validation_key': [0.7160234899328859, -0.0007441272832394699, 0.89200633080671], 'source_files': {'PS_20174392719_1491204439457_log.csv': '16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b'}}, 'sparkov': {'training_cohort': 'bec196feeffe066b782c186eca79d3f2022180facb483e7ea9acf04f53c4bc8b', 'validation_cohort': '03fad280d0f4e1eb221bb822e97d9da4b09f74d5cc3c111721ecf5f7f5055a4d', 'expected_v6_validation_key': [0.6937212863705973, -0.0007448639616697032, 0.808475296623126], 'source_files': {'fraudTrain.csv': 'fd7139200dbfcbed0b6742bbe05a4f1abce532c4fef20918228a651647a3e75d', 'fraudTest.csv': '12d553ab19440c752d2531ee1af44bb64f12cc3d3839f1649f19e81c230545f0'}}, 'baf_base': {'training_cohort': 'ad95770ecebc3298ad639caa99e8d0d48db24781dd3b55693041d423b4944be5', 'validation_cohort': '63b28a96dbb68ea86704963acc8bdc7f2a15ccad6a50b2a8ec3976da7dd59d80', 'expected_v6_validation_key': [0.03897254207263064, -0.000743392905907698, 0.13987847195464725], 'source_files': {'Base.csv': '7bf10a37ce07e72e14c1b09e5efee3d27261baff4facc7da767b0474dcf9b809'}}}, 'holdout_policy': 'All prior final results remain recorded; no V7 final test or independent confirmation in this phase', 'novelty_status': 'Unproven. Fully corrective pAUC ensembles and VPBoost overlap with the proposal and require explicit comparison.'},indent=2))
from v7_research import run
from validation_data import sources
run('baf_base',sources('baf_base',"/kaggle/input"),output,references={'CatBoost_native': {'family': 'CatBoost_native', 'trial': 1, 'config': {'depth': 6, 'positive_weight': 5.0}, 'resolved_positive_weight': 5.0, 'prefix': 65, 'validation_key': [0.05845881310894597, -0.000743392905907698, 0.1588523270101041], 'fit_s': 7.620741411999973, 'selection_s': 9.331869201000018}, 'XGBoost': {'family': 'XGBoost', 'trial': 0, 'config': {'depth': 4, 'positive_weight': 1.0}, 'resolved_positive_weight': 1.0, 'prefix': 70, 'validation_key': [0.053144375553587246, -0.000743392905907698, 0.15730355670542362], 'fit_s': 1.2709206630000267, 'selection_s': 2.3448497499999803}, 'LightGBM_native': {'family': 'LightGBM_native', 'trial': 0, 'config': {'leaves': 15, 'positive_weight': 1.0}, 'resolved_positive_weight': 1.0, 'prefix': 87, 'validation_key': [0.04295837023914969, -0.000743392905907698, 0.14728679923862759], 'fit_s': 1.5658173090000105, 'selection_s': 10.621145109999986}},expected_cohorts={'train': 'ad95770ecebc3298ad639caa99e8d0d48db24781dd3b55693041d423b4944be5', 'validation': '63b28a96dbb68ea86704963acc8bdc7f2a15ccad6a50b2a8ec3976da7dd59d80'},expected_files={'Base.csv': '7bf10a37ce07e72e14c1b09e5efee3d27261baff4facc7da767b0474dcf9b809'})
